# VICTOR Train — v6.0
**WEST SXR Tomography · INR + Dual Hash Grid + SIREN + PIGNO + Ensemble**

| Cell | Contents | Modules |
|------|----------|---------|
| 1 | Install · Config · Module Loader | *(self-contained)* |
| 2 | W matrix · Geometry · Profiles | `geometry.py`, `data_loader.py` |
| 3 | Build model | `model.py` |
| 4 | Build loss functions | `losses.py` |
| 5 | Training loop | `trainer.py`, `checkpoint.py` |
| 6 | Evaluation + plots | `evaluate.py` |

> **How to use:** Run cells in order (1 → 6). All modules are embedded below as `%%writefile` blocks, written to `MODULES_DIR` on first run and auto-reloaded on subsequent runs.

---
## Cell 1 — Install · Config · Module Loader
*(Self-contained. Mounts Drive, installs JAX, defines `load_modules()`.)*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 1: Install + Imports + Config + Module Loader
# ============================================================

# ── 1. JAX install (pinned, version-safe) ───────────────────
import subprocess, sys

def pip(p):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

# Uninstall any mismatched JAX packages from previous sessions
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
                       "jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda12-pjrt"],
                      stderr=subprocess.DEVNULL)

# Reinstall everything pinned to matching versions
JAX_VERSION = "0.4.30"
for p in [
    f"jax[cuda12]=={JAX_VERSION}",
    f"jaxlib=={JAX_VERSION}",
    "flax",
    "optax",
    "scipy",
    "matplotlib",
    "ott-jax",
    "orbax-checkpoint",
]:
    pip(p)

print(f"JAX {JAX_VERSION} + dependencies installed OK")

# ── 2. Mount Drive ──────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE = "/content/drive/MyDrive"
    print("Drive mounted")
except Exception:
    DRIVE = "."
    print("Local mode (Drive not available)")

# ── 3. Core imports ─────────────────────────────────────────
import os, json, pickle, time, functools, importlib
import numpy as np
import scipy.sparse as sp_sci
import matplotlib.pyplot as plt

import jax, jax.numpy as jnp
from jax.experimental.sparse import BCOO
import flax.linen as nn
import optax
from flax.training import train_state

print(f"JAX     : {jax.__version__}")
print(f"Devices : {jax.devices()}")

# ── 4. XLA compilation cache (persists across sessions) ─────
JAX_CACHE = "/content/jax_cache"
os.environ["JAX_COMPILATION_CACHE_DIR"] = JAX_CACHE
os.makedirs(JAX_CACHE, exist_ok=True)
print(f"JIT cache: {JAX_CACHE}")

# ── 5. Project paths ────────────────────────────────────────
DATASET_DIR  = f"{DRIVE}/victor_dataset"
CKPT_DIR     = f"{DRIVE}/VICTOR_v6"
MODULES_DIR  = f"{DRIVE}/victor_modules"      # where all .py modules live
RESULTS_DIR  = "./ono_results"

for d in [CKPT_DIR, MODULES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Dataset  : {DATASET_DIR}")
print(f"Modules  : {MODULES_DIR}")
print(f"Checkpts : {CKPT_DIR}")

# ── 6. Add modules dir to path + autoreload ─────────────────
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)

# Enable autoreload so edits to .py files on Drive take effect immediately
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
        print("Autoreload enabled")
except Exception:
    print("Autoreload not available (non-IPython environment)")

# ── 7. Config constants ─────────────────────────────────────
# Geometry
N_GRID  = 128
EXT     = 0.64
R0      = 2.5
AP      = 0.5
BP      = 0.65

# Hash grid
T_COORD = 8192
T_FIELD = 4096
L_HASH  = 16
F_HASH  = 2

# Ensemble
N_ENS   = 5

# Training
N_EPOCHS   = 10_000
LR         = 3e-4
SAVE_EVERY = 50
LOG_EVERY  = 200

# Noise stages
STAGES = [
    (N_EPOCHS // 3,              0.001),
    (N_EPOCHS // 3,              0.003),
    (N_EPOCHS - 2*(N_EPOCHS//3), 0.008),
]

print(f"\nConfig loaded:")
print(f"  Grid={N_GRID}x{N_GRID}  EXT={EXT}  R0={R0}  AP={AP}  BP={BP}")
print(f"  HashGrid T_coord={T_COORD} T_field={T_FIELD} L={L_HASH} F={F_HASH}")
print(f"  Ensemble N={N_ENS}  Epochs={N_EPOCHS}  LR={LR}")

# ── 8. Module loader ────────────────────────────────────────
# Call load_modules() at the top of any cell to get fresh imports.
# Autoreload handles this automatically during development,
# but load_modules() is useful after a Drive edit mid-session.

_MODULES = {}

def load_modules(force_reload=False):
    """
    Import (or reload) all VICTOR .py modules from Drive.
    Returns a dict of the loaded modules for easy access.

    Usage:
        m = load_modules()
        # or after editing a file:
        m = load_modules(force_reload=True)
    """
    module_names = [
        "config",        # re-exports all constants above as a module
        "geometry",      # build_ray_coords, build_rho_graph, pixel grids
        "data_loader",   # W matrix, profile loading, inject_noise
        "model",         # HashGrid, SIRENLayer, SharedTrunk, VICTOR
        "losses",        # all loss functions, loss_fn, train_step
        "checkpoint",    # save_meta, load_meta, do_checkpoint
        "evaluate",      # metrics, plots
    ]

    loaded = {}
    missing = []

    for name in module_names:
        fpath = os.path.join(MODULES_DIR, f"{name}.py")
        if not os.path.exists(fpath):
            missing.append(name)
            continue
        try:
            if name in _MODULES and not force_reload:
                mod = importlib.reload(_MODULES[name])
            else:
                import importlib.util
                spec = importlib.util.spec_from_file_location(name, fpath)
                mod  = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(mod)
            _MODULES[name] = mod
            loaded[name]   = mod
            print(f"  ✓ {name}")
        except Exception as e:
            print(f"  ✗ {name}  ERROR: {e}")

    if missing:
        print(f"\n  Not yet created: {missing}")
        print(f"  → Save them to {MODULES_DIR}/")

    return loaded

# Try loading whatever modules already exist on Drive
print("\nLoading modules from Drive:")
modules = load_modules()

print("\n" + "="*50)
print("Cell 1 complete — ready to run Cell 2")
print("="*50)


In [ ]:
# ── Embedded module source strings ─────────────────────────────
# These are set once at notebook load time. The write-to-modules
# cell below uses them to populate MODULES_DIR without a Drive edit.

_SRC_config = "# ============================================================\n# VICTOR v6.0 \u2014 config.py\n# All project constants: grid, hash, ensemble, training, paths\n# ============================================================\n# Usage:\n#   import config as cfg\n#   cfg.N_GRID, cfg.R0, cfg.T_COORD, ...\n#\n# All Cell 1 globals are re-exported here so downstream modules\n# only need `import config` instead of relying on notebook globals.\n# ============================================================\n\nimport os\n\n# \u2500\u2500 Grid / geometry \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nN_GRID  = 128       # pixels per side  (128\u00d7128 reconstruction grid)\nEXT     = 0.64      # half-extent [m] of the square pixel domain\nR0      = 2.5       # major radius of grid centre [m]\nAP      = 0.5       # semi-axis in R (ellipse normalisation)\nBP      = 0.65      # semi-axis in Z (ellipse normalisation)\n\n# \u2500\u2500 Hash grid \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nT_COORD = 8192      # hash table size  \u2014 coordinate hash\nT_FIELD = 4096      # hash table size  \u2014 field quantity hash\nL_HASH  = 16        # number of resolution levels\nF_HASH  = 2         # features per level\n\n# \u2500\u2500 Ensemble \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nN_ENS   = 5         # number of ensemble members\n\n# \u2500\u2500 PIGNO \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nPIGNO_LAYERS = 2\nPIGNO_HIDDEN = 96\n\n# \u2500\u2500 Training \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nN_EPOCHS   = 10_000\nLR         = 3e-4\nSAVE_EVERY = 50\nLOG_EVERY  = 200\n\n# Noise stages: list of (n_steps, sigma_fraction) tuples\n# Three progressive noise levels over the full training run\nSTAGES = [\n    (N_EPOCHS // 3,              0.001),\n    (N_EPOCHS // 3,              0.003),\n    (N_EPOCHS - 2*(N_EPOCHS//3), 0.008),\n]\n\n# \u2500\u2500 Profiles \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nN_PROFILES = 10     # number of TORAX equilibrium profiles to load\n\n# \u2500\u2500 rho-graph (PIGNO adjacency) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nRHO_GRAPH_N_NB   = 6      # k-nearest neighbours in rho-space\nRHO_GRAPH_SIGMA  = 0.04   # RBF bandwidth\nRHO_GRAPH_STRIDE = 4      # sub-sample stride for graph building\n\n# \u2500\u2500 Ray tracing (WEST-tuned adaptive march) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Camera specs: (px, py, angle_min_deg, angle_max_deg, n_rays)\nCAMERAS = [\n    (1.5,  0.0, 155., 205., 64),\n    (0.0, -1.5,  65., 115., 64),\n]\nDS_MAX_FACTOR = 0.5        # ds_max = (2*EXT/N_GRID) * DS_MAX_FACTOR\nDS_MIN_FACTOR = 0.5        # ds_min = ds_max * DS_MIN_FACTOR\n\n# \u2500\u2500 Paths \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Populated at runtime by Cell 1; defaults here allow standalone use.\nDRIVE       = os.environ.get(\"VICTOR_DRIVE\", \".\")\nDATASET_DIR = os.environ.get(\"VICTOR_DATASET\",  f\"{DRIVE}/victor_dataset\")\nCKPT_DIR    = os.environ.get(\"VICTOR_CKPT\",     f\"{DRIVE}/VICTOR_v6\")\nMODULES_DIR = os.environ.get(\"VICTOR_MODULES\",  f\"{DRIVE}/victor_modules\")\nRESULTS_DIR = os.environ.get(\"VICTOR_RESULTS\",  \"./ono_results\")\n\n\ndef update_paths(drive: str) -> None:\n    \"\"\"\n    Call once after Drive is mounted (or DRIVE is resolved) to propagate\n    the correct root into all path constants.\n\n    Example (in Cell 2, after loading modules):\n        import config as cfg\n        cfg.update_paths(DRIVE)\n    \"\"\"\n    global DRIVE, DATASET_DIR, CKPT_DIR, MODULES_DIR, RESULTS_DIR\n    DRIVE       = drive\n    DATASET_DIR = f\"{drive}/victor_dataset\"\n    CKPT_DIR    = f\"{drive}/VICTOR_v6\"\n    MODULES_DIR = f\"{drive}/victor_modules\"\n    RESULTS_DIR = \"./ono_results\"\n\n\ndef summary() -> None:\n    \"\"\"Print a compact summary of all active constants.\"\"\"\n    print(\"\u2500\u2500 config.py \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n    print(f\"  Grid     : {N_GRID}\u00d7{N_GRID}  EXT={EXT}  R0={R0}  AP={AP}  BP={BP}\")\n    print(f\"  HashGrid : T_coord={T_COORD}  T_field={T_FIELD}  L={L_HASH}  F={F_HASH}\")\n    print(f\"  Ensemble : N={N_ENS}  Profiles={N_PROFILES}\")\n    print(f\"  Training : epochs={N_EPOCHS}  LR={LR}  save={SAVE_EVERY}  log={LOG_EVERY}\")\n    print(f\"  Stages   : {STAGES}\")\n    print(f\"  Paths    : dataset={DATASET_DIR}\")\n    print(f\"             ckpt={CKPT_DIR}\")\n    print(\"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n\n\nif __name__ == \"__main__\":\n    summary()\n"

_SRC_geometry = "# ============================================================\n# VICTOR v6.0 \u2014 geometry.py\n# Pixel grids, ray coordinates, rho-proximity graph,\n# and W matrix-free matvec / vecmat operators.\n# ============================================================\n# Public API\n# ----------\n#   build_pixel_grids()  \u2192 PixelGrids (namedtuple)\n#   build_ray_coords()   \u2192 RayArrays  (namedtuple)\n#   build_rho_graph()    \u2192 RhoGraph   (namedtuple)\n#   make_W_operators()   \u2192 WOperators (namedtuple)   [called from data_loader]\n#\n# All heavy arrays are returned as JAX arrays (jnp.float32 / jnp.int32).\n# No JAX globals are mutated; callers own the returned objects.\n# ============================================================\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport jax\nimport jax.numpy as jnp\nfrom typing import NamedTuple\n\nimport config as cfg\n\n\n# \u2500\u2500 Named return types \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass PixelGrids(NamedTuple):\n    \"\"\"All per-pixel geometry arrays, shape (N_GRID*N_GRID,) unless noted.\"\"\"\n    RHO_2D   : jnp.ndarray   # (N_GRID, N_GRID)  normalised elliptic radius\n    RHO_FLAT : jnp.ndarray   # (N_GRID\u00b2,)\n    THETA_FLAT: jnp.ndarray  # (N_GRID\u00b2,)  atan2(Y, X)\n    R_PIX    : jnp.ndarray   # (N_GRID\u00b2,)  major radius of each pixel [m]\n    Z_PIX    : jnp.ndarray   # (N_GRID\u00b2,)  vertical position of each pixel [m]\n\n\nclass RayArrays(NamedTuple):\n    \"\"\"Padded ray-march arrays, shape (n_rays, MAX_STEPS).\"\"\"\n    RAY_R     : jnp.ndarray   # major radius  [m]\n    RAY_Z     : jnp.ndarray   # vertical coord [m]\n    RAY_DS    : jnp.ndarray   # step length   [m]\n    RAY_V     : jnp.ndarray   # valid-step mask (0/1)\n    MAX_STEPS : int\n\n\nclass RhoGraph(NamedTuple):\n    \"\"\"Edge lists and weights for the PIGNO rho-proximity graph.\"\"\"\n    EDGES_SRC : jnp.ndarray   # (E,) int32\n    EDGES_DST : jnp.ndarray   # (E,) int32\n    EDGE_W    : jnp.ndarray   # (E,) float32\n    NODE_DEG  : jnp.ndarray   # (N_GRID\u00b2,) float32  \u2265 1\n\n\nclass WOperators(NamedTuple):\n    \"\"\"Matrix-free W operators built from a padded CSR representation.\"\"\"\n    W_IDX : jnp.ndarray    # (128, MX) int32   \u2014 column indices\n    W_LEN : jnp.ndarray    # (128, MX) float32 \u2014 row values\n    W_MSK : jnp.ndarray    # (128, MX) float32 \u2014 valid-entry mask\n    matvec: object          # jit-compiled (ef \u2192 projections)\n    vecmat: object          # jit-compiled (v  \u2192 back-projection)\n\n\n# \u2500\u2500 1. Pixel grids \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef build_pixel_grids(\n    n_grid: int = cfg.N_GRID,\n    ext: float  = cfg.EXT,\n    r0: float   = cfg.R0,\n    ap: float   = cfg.AP,\n    bp: float   = cfg.BP,\n) -> PixelGrids:\n    \"\"\"\n    Build the 128\u00d7128 reconstruction pixel grid.\n\n    Returns\n    -------\n    PixelGrids\n        RHO_2D    : (n_grid, n_grid) normalised elliptic radius \u03c1 = \u221a[(x/ap)\u00b2+(y/bp)\u00b2]\n        RHO_FLAT  : (n_grid\u00b2,)\n        THETA_FLAT: (n_grid\u00b2,)  arctan2(y, x)\n        R_PIX     : (n_grid\u00b2,)  R = r0 + x   [m]\n        Z_PIX     : (n_grid\u00b2,)  Z = y          [m]\n    \"\"\"\n    lin       = np.linspace(-ext, ext, n_grid)\n    XX, YY    = np.meshgrid(lin, lin)\n\n    rho_2d_np   = np.sqrt((XX / ap)**2 + (YY / bp)**2).astype(np.float32)\n    theta_np    = np.arctan2(YY, XX).flatten().astype(np.float32)\n    r_pix_np    = (r0 + XX).flatten().astype(np.float32)\n    z_pix_np    = YY.flatten().astype(np.float32)\n\n    return PixelGrids(\n        RHO_2D    = jnp.array(rho_2d_np),\n        RHO_FLAT  = jnp.array(rho_2d_np.flatten()),\n        THETA_FLAT= jnp.array(theta_np),\n        R_PIX     = jnp.array(r_pix_np),\n        Z_PIX     = jnp.array(z_pix_np),\n    )\n\n\n# \u2500\u2500 2. Ray coordinates (WEST-tuned adaptive march) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef build_ray_coords(\n    cameras          = cfg.CAMERAS,\n    ext: float       = cfg.EXT,\n    n_grid: int      = cfg.N_GRID,\n    ap: float        = cfg.AP,\n    bp: float        = cfg.BP,\n    r0: float        = cfg.R0,\n    ds_max_factor: float = cfg.DS_MAX_FACTOR,\n    ds_min_factor: float = cfg.DS_MIN_FACTOR,\n) -> RayArrays:\n    \"\"\"\n    Compute adaptive-step ray march for all WEST LOS chords.\n\n    The march uses a finer step near the axis (where gradients are\n    steepest) and a coarser step at the plasma boundary.\n\n    Parameters\n    ----------\n    cameras : list of (px, py, angle_min_deg, angle_max_deg, n_rays)\n\n    Returns\n    -------\n    RayArrays\n        RAY_R, RAY_Z, RAY_DS, RAY_V : (n_rays_total, MAX_STEPS)\n        MAX_STEPS                    : int\n    \"\"\"\n    ds_max = (2.0 * ext / n_grid) * ds_max_factor\n    ds_min = ds_max * ds_min_factor\n\n    all_R, all_Z, all_ds = [], [], []\n\n    for px, py, a_min_deg, a_max_deg, nc in cameras:\n        for ang in np.linspace(np.radians(a_min_deg), np.radians(a_max_deg), nc):\n            dx, dy = np.cos(ang), np.sin(ang)\n\n            # Ellipse entry/exit: A t\u00b2 + B t + C = 0\n            A_ = (dx / ap)**2 + (dy / bp)**2\n            B_ = 2.0 * (px * dx / ap**2 + py * dy / bp**2)\n            C_ = (px / ap)**2 + (py / bp)**2 - 1.0\n            disc = B_**2 - 4.0 * A_ * C_\n\n            if disc < 0:\n                all_R.append([])\n                all_Z.append([])\n                all_ds.append([])\n                continue\n\n            t_in  = (-B_ - np.sqrt(disc)) / (2.0 * A_)\n            t_out = (-B_ + np.sqrt(disc)) / (2.0 * A_)\n\n            Rs, Zs, dss = [], [], []\n            t = t_in\n            while t < t_out:\n                xm = px + t * dx\n                ym = py + t * dy\n                rho_loc = np.sqrt((xm / ap)**2 + (ym / bp)**2)\n\n                # Step size: smaller near axis, larger near edge\n                ds = ds_min + (ds_max - ds_min) * (1.0 - min(rho_loc, 1.0))\n                ds = min(ds, t_out - t)\n\n                # Midpoint quadrature\n                xc = px + (t + ds / 2.0) * dx\n                yc = py + (t + ds / 2.0) * dy\n\n                Rs.append(r0 + xc)\n                Zs.append(yc)\n                dss.append(ds)\n                t += ds\n\n            all_R.append(Rs)\n            all_Z.append(Zs)\n            all_ds.append(dss)\n\n    n_rays = len(all_R)\n    mx = max((len(r) for r in all_R), default=1)\n\n    RR = np.zeros((n_rays, mx), dtype=np.float32)\n    ZZ = np.zeros((n_rays, mx), dtype=np.float32)\n    DS = np.zeros((n_rays, mx), dtype=np.float32)\n    VV = np.zeros((n_rays, mx), dtype=np.float32)\n\n    for i in range(n_rays):\n        n = len(all_R[i])\n        if n > 0:\n            RR[i, :n] = all_R[i]\n            ZZ[i, :n] = all_Z[i]\n            DS[i, :n] = all_ds[i]\n            VV[i, :n] = 1.0\n\n    return RayArrays(\n        RAY_R     = jnp.array(RR),\n        RAY_Z     = jnp.array(ZZ),\n        RAY_DS    = jnp.array(DS),\n        RAY_V     = jnp.array(VV),\n        MAX_STEPS = mx,\n    )\n\n\n# \u2500\u2500 3. rho-proximity graph for PIGNO \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef build_rho_graph(\n    rho_flat  : np.ndarray,\n    n_nb      : int   = cfg.RHO_GRAPH_N_NB,\n    sigma     : float = cfg.RHO_GRAPH_SIGMA,\n    stride    : int   = cfg.RHO_GRAPH_STRIDE,\n) -> RhoGraph:\n    \"\"\"\n    Build a k-nearest-neighbour graph in rho-space for the PIGNO layers.\n\n    Uses sub-sampling (stride) to keep edge count tractable, then\n    weights edges by an RBF kernel exp(-|\u03c1\u1d62 - \u03c1\u2c7c| / \u03c3).\n\n    Parameters\n    ----------\n    rho_flat : (N,) numpy array of normalised radii\n    n_nb     : number of neighbours per node\n    sigma    : RBF bandwidth in \u03c1-space\n    stride   : sub-sample nodes every `stride` pixels\n\n    Returns\n    -------\n    RhoGraph\n        EDGES_SRC, EDGES_DST : (E,) int32\n        EDGE_W               : (E,) float32\n        NODE_DEG             : (N,) float32  \u2265 1\n    \"\"\"\n    rho = np.array(rho_flat)\n    N   = len(rho)\n    idx = np.arange(0, N, stride)\n\n    src_list, dst_list, w_list = [], [], []\n\n    for i in idx:\n        ri  = rho[i]\n        dr  = np.abs(rho[idx] - ri)\n        w   = np.exp(-dr / sigma)\n        top = np.argsort(w)[-(n_nb + 1):]   # highest weights (incl. self)\n        for j2 in top:\n            j = idx[j2]\n            if j != i and w[j2] > 0.05:\n                src_list.append(i)\n                dst_list.append(j)\n                w_list.append(float(w[j2]))\n\n    # Node degree (for normalisation inside PIGNO)\n    deg = np.zeros(N, dtype=np.float32)\n    for d_ in dst_list:\n        deg[d_] += 1.0\n\n    return RhoGraph(\n        EDGES_SRC = jnp.array(src_list, dtype=jnp.int32),\n        EDGES_DST = jnp.array(dst_list, dtype=jnp.int32),\n        EDGE_W    = jnp.array(w_list,   dtype=jnp.float32),\n        NODE_DEG  = jnp.array(np.maximum(deg, 1.0)),\n    )\n\n\n# \u2500\u2500 4. W matrix-free operators \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef make_W_operators(W_csr) -> WOperators:\n    \"\"\"\n    Build padded CSR arrays and JIT-compile W\u00b7\u03b5 and W\u1d40\u00b7v operators.\n\n    The padded representation avoids Python loops at inference time:\n    every row is padded to MX (the max nnz per row), controlled by a\n    binary mask W_MSK.\n\n    Parameters\n    ----------\n    W_csr : scipy.sparse.csr_matrix\n        Row-normalised W matrix (128 \u00d7 N_PIXELS).\n\n    Returns\n    -------\n    WOperators\n        W_IDX, W_LEN, W_MSK : JAX arrays\n        matvec               : jit fn  ef \u2192 (128,)\n        vecmat               : jit fn  v  \u2192 (N_PIXELS,)\n    \"\"\"\n    n_rows = W_csr.shape[0]\n    MX     = int(np.diff(W_csr.indptr).max())\n\n    IDX_ = np.zeros((n_rows, MX), dtype=np.int32)\n    LEN_ = np.zeros((n_rows, MX), dtype=np.float32)\n    MSK_ = np.zeros((n_rows, MX), dtype=np.float32)\n\n    for i in range(n_rows):\n        start, end = W_csr.indptr[i], W_csr.indptr[i + 1]\n        n = end - start\n        IDX_[i, :n] = W_csr.indices[start:end]\n        LEN_[i, :n] = W_csr.data[start:end]\n        MSK_[i, :n] = 1.0\n\n    W_IDX = jnp.array(IDX_)\n    W_LEN = jnp.array(LEN_)\n    W_MSK = jnp.array(MSK_)\n\n    @jax.jit\n    def matvec(ef):\n        \"\"\"W \u00b7 ef   \u2192   (n_rows,)  sinogram projection.\"\"\"\n        return jnp.sum(ef[W_IDX] * W_LEN * W_MSK, axis=1)\n\n    @jax.jit\n    def vecmat(v):\n        \"\"\"W\u1d40 \u00b7 v   \u2192   (N_PIXELS,)  adjoint back-projection.\"\"\"\n        n_pixels = W_csr.shape[1]\n        return (\n            jnp.zeros(n_pixels)\n            .at[W_IDX.ravel()]\n            .add((v[:, None] * W_LEN * W_MSK).ravel())\n        )\n\n    return WOperators(\n        W_IDX  = W_IDX,\n        W_LEN  = W_LEN,\n        W_MSK  = W_MSK,\n        matvec = matvec,\n        vecmat = vecmat,\n    )\n\n\n# \u2500\u2500 Convenience: build everything in one call \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef build_all_geometry(W_csr=None):\n    \"\"\"\n    Build pixel grids, ray coords, and rho-graph.\n    Optionally also build W operators if W_csr is provided.\n\n    Returns\n    -------\n    grids    : PixelGrids\n    rays     : RayArrays\n    rho_graph: RhoGraph\n    w_ops    : WOperators | None\n    \"\"\"\n    print(\"Building pixel grids...\")\n    grids = build_pixel_grids()\n    print(f\"  RHO_2D={grids.RHO_2D.shape}  R_PIX={grids.R_PIX.shape}\")\n\n    print(\"Building WEST ray coords...\")\n    rays = build_ray_coords()\n    n_rays = rays.RAY_R.shape[0]\n    mem_kb = n_rays * rays.MAX_STEPS * 4 * 4 / 1024\n    print(f\"  Rays={n_rays}  MaxSteps={rays.MAX_STEPS}  Mem\u2248{mem_kb:.0f} KB\")\n\n    print(\"Building rho-proximity graph...\")\n    rho_graph = build_rho_graph(np.array(grids.RHO_FLAT))\n    print(f\"  Edges={len(rho_graph.EDGES_SRC)}\")\n\n    w_ops = None\n    if W_csr is not None:\n        print(\"Building W matrix-free operators...\")\n        w_ops = make_W_operators(W_csr)\n        print(f\"  W_IDX={w_ops.W_IDX.shape}  MX={w_ops.W_IDX.shape[1]}\")\n\n    return grids, rays, rho_graph, w_ops\n"

_SRC_data_loader = "# ============================================================\n# VICTOR v6.0 \u2014 data_loader.py\n# W matrix loading, TORAX profile loading, noise injection,\n# and field interpolation onto the reconstruction pixel grid.\n# ============================================================\n# Public API\n# ----------\n#   load_W_matrix(path)              \u2192 WBundle (namedtuple)\n#   load_profiles(dataset_dir, ...)  \u2192 list[dict]\n#   inject_noise(g, sigma, key)      \u2192 jnp.ndarray\n#   interp_field(field, R_from, Z_from, R_to, Z_to) \u2192 np.ndarray\n#\n# Design principles\n# -----------------\n#  \u2022 Profile dicts contain plain jnp arrays \u2014 never passed as\n#    dict items into @jax.jit (FIX from v5).\n#  \u2022 W operators come from geometry.make_W_operators(); this\n#    module only handles I/O (load, normalise, report).\n#  \u2022 Field interpolation is NumPy / SciPy (runs once at load time).\n# ============================================================\n\nfrom __future__ import annotations\n\nimport os\nimport numpy as np\nimport scipy.sparse as sp_sci\nfrom scipy.sparse import diags as spd\nfrom scipy.interpolate import RegularGridInterpolator\nfrom typing import NamedTuple, List, Dict, Any, Optional\n\nimport jax\nimport jax.numpy as jnp\nfrom jax.experimental.sparse import BCOO\n\nimport config as cfg\nimport geometry as geom\n\n\n# \u2500\u2500 Named return type for the W bundle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass WBundle(NamedTuple):\n    \"\"\"Everything derived from the W matrix file.\"\"\"\n    W_norm      : Any           # scipy.sparse.csr_matrix  (row-normalised)\n    W_BCOO      : jnp.ndarray   # JAX BCOO sparse tensor\n    ACTIVE_MASK : jnp.ndarray   # (128,) bool  \u2014 rows with non-zero sum\n    ACTIVE_MASK_NP: np.ndarray  # same, as numpy (for Python-side indexing)\n    N_ACTIVE    : int\n    w_ops       : geom.WOperators\n\n\n# \u2500\u2500 1. Load & normalise W matrix \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_W_matrix(path: Optional[str] = None) -> WBundle:\n    \"\"\"\n    Load the W projection matrix, row-normalise, build BCOO and\n    matrix-free operators.\n\n    Parameters\n    ----------\n    path : str, optional\n        Full path to W_matrix.npz.\n        Defaults to ``cfg.DATASET_DIR/W_matrix.npz``.\n\n    Returns\n    -------\n    WBundle\n    \"\"\"\n    if path is None:\n        path = os.path.join(cfg.DATASET_DIR, \"W_matrix.npz\")\n\n    print(f\"Loading W matrix from {path} ...\")\n    W_sp  = sp_sci.load_npz(path).tocsr()\n\n    # Row sums \u2192 active-chord mask\n    rs              = np.array(W_sp.sum(axis=1)).flatten()\n    ACTIVE_MASK_NP  = (rs > 1e-8)\n    rs_safe         = np.where(ACTIVE_MASK_NP, rs, 1.0)\n\n    # Row-normalise (each chord sums to 1)\n    W_norm          = spd(1.0 / rs_safe) @ W_sp\n    ACTIVE_MASK     = jnp.array(ACTIVE_MASK_NP)\n    N_ACTIVE        = int(ACTIVE_MASK_NP.sum())\n\n    # BCOO sparse tensor (for reference / loss debugging)\n    Wcoo   = W_norm.tocoo()\n    W_BCOO = BCOO(\n        (\n            jnp.array(Wcoo.data,  dtype=jnp.float32),\n            jnp.array(np.stack([Wcoo.row, Wcoo.col], axis=1), dtype=jnp.int32),\n        ),\n        shape=W_norm.shape,\n    )\n\n    # Matrix-free operators (JIT compiled)\n    w_ops = geom.make_W_operators(W_norm.tocsr())\n\n    print(\n        f\"  W: {W_norm.shape}  NNZ={W_norm.nnz}  \"\n        f\"active={N_ACTIVE}/{W_sp.shape[0]}\"\n    )\n\n    return WBundle(\n        W_norm        = W_norm,\n        W_BCOO        = W_BCOO,\n        ACTIVE_MASK   = ACTIVE_MASK,\n        ACTIVE_MASK_NP= ACTIVE_MASK_NP,\n        N_ACTIVE      = N_ACTIVE,\n        w_ops         = w_ops,\n    )\n\n\n# \u2500\u2500 2. Field interpolation helper \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef interp_field(\n    field  : np.ndarray,\n    R_from : np.ndarray,\n    Z_from : np.ndarray,\n    R_to   : np.ndarray,\n    Z_to   : np.ndarray,\n) -> np.ndarray:\n    \"\"\"\n    Bilinear interpolation of a 2-D field from one (R,Z) grid to another.\n\n    Parameters\n    ----------\n    field  : (nR, nZ)  source field values\n    R_from : (nR,)     source R axis\n    Z_from : (nZ,)     source Z axis\n    R_to   : (nR2, nZ2) target R coordinates\n    Z_to   : (nR2, nZ2) target Z coordinates\n\n    Returns\n    -------\n    np.ndarray of shape R_to.shape, dtype float32\n        Out-of-bounds points are filled with 0.\n    \"\"\"\n    fn  = RegularGridInterpolator(\n        (R_from, Z_from), field,\n        method=\"linear\",\n        bounds_error=False,\n        fill_value=0.0,\n    )\n    pts = np.stack([R_to.flatten(), Z_to.flatten()], axis=1)\n    return fn(pts).reshape(R_to.shape).astype(np.float32)\n\n\n# \u2500\u2500 3. Noise injection \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef inject_noise(\n    g     : jnp.ndarray,\n    sigma : float,\n    key   : jnp.ndarray,\n    ps    : float = 1e5,\n) -> jnp.ndarray:\n    \"\"\"\n    Add Poisson + Gaussian noise to a sinogram measurement.\n\n    The Poisson term models photon statistics; the Gaussian term\n    models electronic / systematic noise scaled to the peak signal.\n\n    Parameters\n    ----------\n    g     : (128,)  clean sinogram  [a.u.]\n    sigma : float   Gaussian noise fraction of peak signal\n    key   : JAX PRNG key\n    ps    : float   photon scale factor (higher \u2192 less Poisson noise)\n\n    Returns\n    -------\n    jnp.ndarray  (128,)  noisy sinogram, clipped to \u2265 0\n    \"\"\"\n    k_poisson, k_gaussian = jax.random.split(key)\n    lam = jnp.maximum(g * ps, 1.0)\n    g_poisson = jnp.maximum(\n        (lam + jnp.sqrt(lam) * jax.random.normal(k_poisson, lam.shape)) / ps,\n        0.0,\n    )\n    return jnp.maximum(\n        g_poisson + sigma * jnp.max(jnp.abs(g)) * jax.random.normal(k_gaussian, g.shape),\n        0.0,\n    )\n\n\n# \u2500\u2500 4. Field normalisation helper \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _safe_norm_11(x: np.ndarray) -> np.ndarray:\n    \"\"\"Linearly rescale array to [-1, 1] (safe against zero-range).\"\"\"\n    mn, mx = x.min(), x.max()\n    return (2.0 * (x - mn) / (mx - mn + 1e-8) - 1.0).astype(np.float32)\n\n\n# \u2500\u2500 5. Load TORAX equilibrium profiles \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_profiles(\n    dataset_dir  : Optional[str]       = None,\n    n_profiles   : int                  = cfg.N_PROFILES,\n    w_bundle     : Optional[WBundle]    = None,\n    grids        : Optional[geom.PixelGrids] = None,\n) -> List[Dict[str, Any]]:\n    \"\"\"\n    Load TORAX equilibrium profiles and pre-process for training.\n\n    Each profile dict contains **plain jnp arrays** (never nested dicts\n    passed into @jax.jit), plus numpy arrays for Python-side diagnostics.\n\n    Parameters\n    ----------\n    dataset_dir : str, optional\n        Directory containing profile_XXXX.npz files.\n        Defaults to ``cfg.DATASET_DIR``.\n    n_profiles  : int\n        Number of profiles to load.\n    w_bundle    : WBundle\n        Used to compute ideal sinogram g_ideal = W \u00b7 \u03b5.\n        If None, g_ideal is set to zeros.\n    grids       : PixelGrids\n        Pixel grid arrays for field interpolation.\n        If None, built internally via geometry.build_pixel_grids().\n\n    Returns\n    -------\n    list of dicts, each containing:\n        # \u2500\u2500 JAX arrays (safe to index in JIT via positional args) \u2500\u2500\n        psi_n    : (N_GRID\u00b2,) float32   normalised \u03c8  field  \u2208 [-1,1]\n        bpol_n   : (N_GRID\u00b2,) float32   normalised Bpol field \u2208 [-1,1]\n        rho_1d   : (n_rho,)   float32   TORAX 1-D rho axis\n        Te_1d    : (n_rho,)   float32   electron temperature [keV]\n        ne_1d    : (n_rho,)   float32   electron density\n        g_ideal  : (128,)     float32   clean projected sinogram\n\n        # \u2500\u2500 Scalars \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        g_scale  : float   max of raw sinogram (for de-normalisation)\n        idx      : int     profile index\n\n        # \u2500\u2500 NumPy (monitoring / GS loss, never in JIT) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        eps_n       : (N_GRID, N_GRID) float32  normalised emissivity\n        psi_2d_np   : (nR, nZ)         float32  raw \u03c8 on eq-grid\n        R_grid_np   : (nR,)            float32  R axis of eq-grid\n        Z_grid_np   : (nZ,)            float32  Z axis of eq-grid\n    \"\"\"\n    if dataset_dir is None:\n        dataset_dir = cfg.DATASET_DIR\n\n    if grids is None:\n        grids = geom.build_pixel_grids()\n\n    R_pix_np = np.array(grids.R_PIX).reshape(cfg.N_GRID, cfg.N_GRID)\n    Z_pix_np = np.array(grids.Z_PIX).reshape(cfg.N_GRID, cfg.N_GRID)\n\n    # W matvec function (or a no-op if w_bundle not supplied)\n    if w_bundle is not None:\n        _W_matvec = w_bundle.w_ops.matvec\n    else:\n        def _W_matvec(ef):\n            return jnp.zeros(128, dtype=jnp.float32)\n\n    profiles: List[Dict[str, Any]] = []\n    print(f\"\\nLoading {n_profiles} profiles from {dataset_dir}/\")\n\n    for profile_idx in range(n_profiles):\n        fp = os.path.join(dataset_dir, f\"profile_{profile_idx:04d}.npz\")\n\n        if not os.path.exists(fp):\n            print(f\"  MISSING {fp}\")\n            continue\n\n        d = np.load(fp, allow_pickle=True)\n\n        # \u2500\u2500 Emissivity (target) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        eps_raw = d[\"epsilon_2d\"].astype(np.float32)\n        eps_max = float(eps_raw.max()) + 1e-10\n        eps_n   = eps_raw / eps_max                              # (N_GRID, N_GRID)\n\n        # Ideal sinogram\n        g_raw   = _W_matvec(jnp.array(eps_n.flatten()))\n        g_max   = float(g_raw.max()) + 1e-10\n        g_ideal = g_raw / g_max\n\n        # \u2500\u2500 Equilibrium fields (eq-grid \u2192 recon-grid) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        R_g      = d[\"R_grid\"].astype(np.float32)\n        Z_g      = d[\"Z_grid\"].astype(np.float32)\n        psi_raw  = d[\"psi_2d\"].astype(np.float32)\n        Bpol_raw = d[\"B_pol\"].astype(np.float32)\n\n        psi_grid  = interp_field(psi_raw,  R_g, Z_g, R_pix_np, Z_pix_np)\n        bpol_grid = interp_field(Bpol_raw, R_g, Z_g, R_pix_np, Z_pix_np)\n\n        # \u2500\u2500 1-D TORAX profiles \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        rho_1d = d[\"rho\"].astype(np.float32)\n        Te_1d  = d[\"Te\"].astype(np.float32)\n        ne_1d  = d[\"ne\"].astype(np.float32)\n\n        profiles.append(dict(\n            # \u2500\u2500 JIT-safe JAX arrays (positional args to forward pass) \u2500\n            psi_n   = jnp.array(_safe_norm_11(psi_grid).flatten()),    # (N\u00b2,)\n            bpol_n  = jnp.array(_safe_norm_11(bpol_grid).flatten()),   # (N\u00b2,)\n            rho_1d  = jnp.array(rho_1d),\n            Te_1d   = jnp.array(Te_1d),\n            ne_1d   = jnp.array(ne_1d),\n            g_ideal = g_ideal,\n\n            # \u2500\u2500 Scalars \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            g_scale = g_max,\n            idx     = profile_idx,\n\n            # \u2500\u2500 NumPy arrays (monitoring / GS loss, not in JIT) \u2500\u2500\u2500\u2500\u2500\u2500\n            eps_n       = eps_n,\n            psi_2d_np   = psi_raw,\n            R_grid_np   = R_g,\n            Z_grid_np   = Z_g,\n        ))\n\n        print(\n            f\"  profile_{profile_idx:04d}: \"\n            f\"Te_core={float(Te_1d[0]):.2f} keV  \"\n            f\"g_max={g_max:.4f}\"\n        )\n\n    print(f\"\\nLoaded {len(profiles)}/{n_profiles} profiles OK\")\n    return profiles\n\n\n# \u2500\u2500 6. Convenience: load everything for Cell 2 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_cell2(\n    dataset_dir: Optional[str] = None,\n) -> tuple:\n    \"\"\"\n    Full Cell 2 data pipeline in one call.\n\n    Returns\n    -------\n    w_bundle : WBundle\n    grids    : PixelGrids\n    rays     : RayArrays\n    rho_graph: RhoGraph\n    profiles : list[dict]\n    \"\"\"\n    if dataset_dir is None:\n        dataset_dir = cfg.DATASET_DIR\n\n    # W matrix\n    w_bundle = load_W_matrix(os.path.join(dataset_dir, \"W_matrix.npz\"))\n\n    # Geometry\n    grids, rays, rho_graph, _ = geom.build_all_geometry(\n        W_csr=None  # W operators already built inside load_W_matrix\n    )\n\n    # Profiles\n    profiles = load_profiles(\n        dataset_dir = dataset_dir,\n        n_profiles  = cfg.N_PROFILES,\n        w_bundle    = w_bundle,\n        grids       = grids,\n    )\n\n    return w_bundle, grids, rays, rho_graph, profiles\n"

_SRC_model = "# ============================================================\n# VICTOR v6.0 \u2014 model.py\n# Architecture: HashGrid, SIRENLayer, SO2Harmonics,\n#               SharedTrunk, PIGNO, VICTOR_v6\n# ============================================================\n# Public API\n# ----------\n#   HashGrid       \u2014 multi-resolution hash grid (pure JAX)\n#   SIRENLayer     \u2014 SIREN layer with learnable \u03c9\u2080\n#   SO2Harmonics   \u2014 SO(2) angular harmonics encoder\n#   SharedTrunk    \u2014 shared feature extractor (hash + SIREN)\n#   MemberAdapter  \u2014 per-ensemble-member adapter MLP\n#   PIGNO          \u2014 physics-informed graph neural operator\n#   VICTOR_v6      \u2014 full model: shared trunk + adapters + PIGNO\n#\n#   build_model()  \u2014 instantiate VICTOR_v6 and initialise params\n#   count_params() \u2014 count total trainable parameters\n#\n# Design principles\n# -----------------\n#  \u2022 All modules are pure Flax nn.Module subclasses.\n#  \u2022 No JAX globals are mutated; callers own the returned params.\n#  \u2022 profile field arrays (psi_n, bpol_n) are plain jnp arrays \u2014\n#    never nested dicts passed into @jax.jit (v5 FIX).\n#  \u2022 PIGNO boundary mask is a smooth sigmoid (hard boundary via \u03c1).\n# ============================================================\n\nfrom __future__ import annotations\n\nfrom typing import List, Tuple, NamedTuple\n\nimport jax\nimport jax.numpy as jnp\nimport flax.linen as nn\n\nimport config as cfg\n\n\n# \u2500\u2500 Named return type \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass ModelBundle(NamedTuple):\n    \"\"\"Model instance together with its initialised parameters.\"\"\"\n    model  : nn.Module\n    params : dict\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 1.  HashGrid\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass HashGrid(nn.Module):\n    \"\"\"\n    Multi-resolution hash grid encoding (pure JAX \u2014 no CUDA extensions).\n\n    Maps a batch of 2-D coordinates in [-1, 1]\u00b2 to a concatenated\n    feature vector of length L \u00d7 F by bilinearly interpolating\n    per-level hash tables.\n\n    Parameters\n    ----------\n    L : int   Number of resolution levels.\n    T : int   Hash-table size (number of entries per level).\n    F : int   Feature dimension per entry.\n    b : float Growth factor between consecutive level resolutions.\n\n    Input\n    -----\n    xy : (N, 2)  coordinates in [-1, 1]\u00b2\n\n    Output\n    ------\n    (N, L*F)  concatenated multi-scale features\n    \"\"\"\n\n    L : int   = cfg.L_HASH\n    T : int   = cfg.T_COORD\n    F : int   = cfg.F_HASH\n    b : float = 1.38\n\n    # Fibonacci hash constants (int32 two's-complement safe)\n    _PI1 : int = -1_640_531_535\n    _PI2 : int =    805_459_861\n\n    @nn.compact\n    def __call__(self, xy: jnp.ndarray) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        xy : (N, 2)  float32, values in [-1, 1]\n\n        Returns\n        -------\n        (N, L*F)  float32\n        \"\"\"\n        pi1 = jnp.int32(self._PI1)\n        pi2 = jnp.int32(self._PI2)\n\n        feats: List[jnp.ndarray] = []\n\n        for lev in range(self.L):\n            res   = int(16 * (self.b ** lev))\n            table = self.param(\n                f\"t{lev}\",\n                nn.initializers.uniform(1e-4),\n                (self.T, self.F),\n            )\n\n            # Map to [0, res]\n            sc = (xy * 0.5 + 0.5) * res\n            fl = jnp.floor(sc).astype(jnp.int32)\n            fr = sc - fl.astype(jnp.float32)           # fractional part\n\n            # Four corners of the enclosing cell (N, 4, 2)\n            corners = jnp.stack(\n                [\n                    fl,\n                    fl + jnp.array([1, 0]),\n                    fl + jnp.array([0, 1]),\n                    fl + jnp.array([1, 1]),\n                ],\n                axis=1,\n            )\n\n            cx = corners[:, :, 0]   # (N, 4)\n            cy = corners[:, :, 1]   # (N, 4)\n\n            # Spatial hash: h = |(cx * \u03c0\u2081) XOR (cy * \u03c0\u2082)| mod T\n            h = jnp.abs(\n                jnp.mod(\n                    jnp.bitwise_xor(\n                        (cx * pi1).astype(jnp.int32),\n                        (cy * pi2).astype(jnp.int32),\n                    ),\n                    self.T,\n                )\n            )\n\n            cf = table[h]           # (N, 4, F)\n\n            # Bilinear weights (N, 4, 1)\n            fx = fr[:, 0:1]\n            fy = fr[:, 1:2]\n            w  = jnp.stack(\n                [(1 - fx) * (1 - fy), fx * (1 - fy),\n                 (1 - fx) * fy,       fx * fy],\n                axis=1,\n            )\n\n            feats.append(jnp.sum(w * cf, axis=1))   # (N, F)\n\n        return jnp.concatenate(feats, axis=-1)        # (N, L*F)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 2.  SIRENLayer\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass SIRENLayer(nn.Module):\n    \"\"\"\n    Sinusoidal representation network (SIREN) layer with a\n    learnable per-layer frequency parameter \u03c9\u2080.\n\n    The first layer uses a uniform weight initialisation over\n    (-1/fan_in, 1/fan_in); subsequent layers use the SIREN-paper\n    scheme \u221a(6/fan_in) / \u03c9\u2080.\n\n    Parameters\n    ----------\n    features  : int    Output width.\n    is_first  : bool   True for the first layer of a SIREN stack.\n    omega_0   : float  Initial value of the learnable \u03c9\u2080.\n    \"\"\"\n\n    features : int\n    is_first : bool  = False\n    omega_0  : float = 30.0\n\n    @nn.compact\n    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        x : (N, D)  float32\n\n        Returns\n        -------\n        (N, features)  float32   sin(\u03c9\u2080 \u00b7 W x)\n        \"\"\"\n        om = self.param(\n            \"om\",\n            nn.initializers.constant(self.omega_0),\n            (1,),\n        )\n\n        if self.is_first:\n            scale = 1.0 / x.shape[-1]\n        else:\n            scale = jnp.sqrt(6.0 / x.shape[-1]) / self.omega_0\n\n        w = nn.Dense(\n            self.features,\n            kernel_init=nn.initializers.uniform(scale),\n        )(x)\n\n        return jnp.sin(om * w)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 3.  SO2Harmonics\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass SO2Harmonics(nn.Module):\n    \"\"\"\n    SO(2) angular harmonic encoder.\n\n    Computes cos(m\u00b7\u03b8) and sin(m\u00b7\u03b8) for m = 0 \u2026 n_orders-1, then\n    projects the raw harmonics through a learned Dense layer.\n\n    Parameters\n    ----------\n    n_orders : int   Number of angular frequency orders.\n\n    Input\n    -----\n    theta : (N,)  angles in radians\n\n    Output\n    ------\n    (N, n_orders*2)  float32\n    \"\"\"\n\n    n_orders : int = 4\n\n    @nn.compact\n    def __call__(self, theta: jnp.ndarray) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        theta : (N,)  float32\n\n        Returns\n        -------\n        (N, n_orders*2)  float32\n        \"\"\"\n        parts = [jnp.ones_like(theta[:, None])]   # m = 0  (constant)\n\n        for m in range(1, self.n_orders):\n            parts.append(jnp.cos(m * theta[:, None]))\n            parts.append(jnp.sin(m * theta[:, None]))\n\n        h = jnp.concatenate(parts, axis=-1)        # (N, 2*n_orders - 1)\n\n        return nn.Dense(\n            self.n_orders * 2,\n            kernel_init=nn.initializers.glorot_normal(),\n        )(h)                                        # (N, n_orders*2)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 4.  SharedTrunk\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass SharedTrunk(nn.Module):\n    \"\"\"\n    Shared feature extractor \u2014 parameters shared across all ensemble\n    members.\n\n    Concatenates spatial hash features, field hash features, and SO(2)\n    angular features, then passes them through a two-layer SIREN MLP.\n\n    Input dimensions (defaults)\n    ---------------------------\n    hash_coord : (N, L*F)     = (N, 32)   spatial hash\n    hash_field : (N, L//2*F)  = (N, 16)   field hash\n    so2_feat   : (N, 8)                   angular harmonics\n    \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    concat       (N, 56)\n\n    Output\n    ------\n    (N, hidden)  float32   shared trunk features\n    \"\"\"\n\n    hidden : int = 256\n\n    @nn.compact\n    def __call__(\n        self,\n        hash_coord : jnp.ndarray,\n        hash_field : jnp.ndarray,\n        so2_feat   : jnp.ndarray,\n    ) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        hash_coord : (N, 32)\n        hash_field : (N, 16)\n        so2_feat   : (N, 8)\n\n        Returns\n        -------\n        (N, hidden)  float32\n        \"\"\"\n        x = jnp.concatenate([hash_coord, hash_field, so2_feat], axis=-1)\n        x = SIRENLayer(self.hidden, is_first=True)(x)\n        x = SIRENLayer(self.hidden)(x)\n        return x                                    # (N, hidden)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 5.  MemberAdapter\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass MemberAdapter(nn.Module):\n    \"\"\"\n    Tiny per-ensemble-member adapter MLP.\n\n    Each adapter has independent weights and maps the shared trunk\n    output to a positive scalar emissivity at every pixel.\n\n    Architecture:  trunk \u2192 Dense(64) \u2192 GELU \u2192 Dense(1) \u2192 softplus\n\n    Input\n    -----\n    trunk_out : (N, 256)   shared trunk features\n\n    Output\n    ------\n    (N,)  float32   positive emissivity prediction for one member\n    \"\"\"\n\n    @nn.compact\n    def __call__(self, trunk_out: jnp.ndarray) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        trunk_out : (N, hidden)\n\n        Returns\n        -------\n        (N,)  float32  softplus-activated scalar output\n        \"\"\"\n        h   = nn.Dense(64, kernel_init=nn.initializers.glorot_normal())(trunk_out)\n        h   = nn.gelu(h)\n        out = nn.Dense(1,  kernel_init=nn.initializers.normal(0.01))(h)\n        return nn.softplus(out.squeeze(-1))          # (N,)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 6.  PIGNO  \u2014  Physics-Informed Graph Neural Operator\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass PIGNO(nn.Module):\n    \"\"\"\n    Physics-Informed Graph Neural Operator.\n\n    Performs message passing on the rho-proximity graph to propagate\n    information along iso-flux surfaces, enforcing the physical\n    constraint that emissivity varies smoothly with normalised radius \u03c1.\n\n    Each layer computes edge messages from (h_src, h_dst, weight),\n    aggregates by degree-normalised scatter, and applies LayerNorm\n    with a residual connection.\n\n    Parameters\n    ----------\n    n_layers : int   Number of message-passing layers (default 2).\n    hidden   : int   Hidden width in the edge MLP (default 96).\n    N        : int   Grid size; output is (N, N) (default 128).\n\n    Inputs\n    ------\n    eps_2d : (N, N)   initial emissivity estimate from the ensemble mean\n    esrc   : (E,)     int32  edge source indices  (flat pixel index)\n    edst   : (E,)     int32  edge destination indices\n    ew     : (E,)     float32 edge weights (RBF in \u03c1-space)\n    ndeg   : (N\u00b2,)   float32 per-node degree (\u2265 1)\n\n    Output\n    ------\n    (N, N)  float32   refined emissivity, positive via softplus\n    \"\"\"\n\n    n_layers : int = cfg.PIGNO_LAYERS\n    hidden   : int = cfg.PIGNO_HIDDEN\n    N        : int = cfg.N_GRID\n\n    @nn.compact\n    def __call__(\n        self,\n        eps_2d : jnp.ndarray,\n        esrc   : jnp.ndarray,\n        edst   : jnp.ndarray,\n        ew     : jnp.ndarray,\n        ndeg   : jnp.ndarray,\n    ) -> jnp.ndarray:\n        \"\"\"\n        Parameters\n        ----------\n        eps_2d : (N, N)\n        esrc   : (E,) int32\n        edst   : (E,) int32\n        ew     : (E,) float32\n        ndeg   : (N\u00b2,) float32\n\n        Returns\n        -------\n        (N, N)  float32\n        \"\"\"\n        n_nodes = self.N * self.N\n        h       = eps_2d.flatten()[:, None]          # (N\u00b2, 1)\n        deg     = ndeg[:, None]                      # (N\u00b2, 1)\n\n        for _ in range(self.n_layers):\n            h_src  = h[esrc]                         # (E, 1)\n            h_dst  = h[edst]                         # (E, 1)\n            w      = ew[:, None]                     # (E, 1)\n\n            # Edge message: concat [h_src | h_dst | weight]\n            msg = jnp.concatenate([h_src, h_dst, w], axis=-1)   # (E, 3)\n            msg = nn.Dense(\n                self.hidden,\n                kernel_init=nn.initializers.glorot_normal(),\n            )(msg)\n            msg = nn.gelu(msg)\n            msg = nn.Dense(\n                1,\n                kernel_init=nn.initializers.glorot_normal(),\n            )(msg)                                               # (E, 1)\n\n            # Degree-normalised scatter-add to destination nodes\n            h_agg = (\n                jnp.zeros((n_nodes, 1))\n                .at[edst]\n                .add(msg * w)\n            )\n            h_agg = h_agg / deg                      # normalise\n\n            # Residual + LayerNorm\n            h = nn.LayerNorm()(h + h_agg)\n\n        return nn.softplus(h).squeeze(-1).reshape(self.N, self.N)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 7.  VICTOR_v6  \u2014  Full Model\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass VICTOR_v6(nn.Module):\n    \"\"\"\n    VICTOR v6.0 \u2014 full reconstruction model.\n\n    Architecture summary\n    --------------------\n    \u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510\n    \u2502  Dual hash grid                                      \u2502\n    \u2502    coord hash  (T=8192, L=16, F=2) \u2192 (N, 32)        \u2502\n    \u2502    field hash  (T=4096, L= 8, F=2) \u2192 (N, 16)        \u2502\n    \u2502  SO2Harmonics  (n_orders=4)        \u2192 (N,  8)        \u2502\n    \u2502                                    \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500        \u2502\n    \u2502  Concat                            \u2192 (N, 56)        \u2502\n    \u2502  SharedTrunk  (hidden=256)         \u2192 (N,256)        \u2502\n    \u2502                                                     \u2502\n    \u2502  MemberAdapter \u00d7 n_members         \u2192 (M, N)         \u2502\n    \u2502    mean / std of ensemble predictions               \u2502\n    \u2502                                                     \u2502\n    \u2502  PIGNO  (n_layers=2, hidden=96)                     \u2502\n    \u2502    applied to ensemble mean (128,128)               \u2502\n    \u2502                                                     \u2502\n    \u2502  Sigmoid boundary mask (\u03c1 < 1)                      \u2502\n    \u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518\n\n    Total ~ 570 k parameters (target 500 k \u2013 600 k).\n\n    Forward signature\n    -----------------\n    (R_flat, Z_flat, psi_n, bpol_n, esrc, edst, ew, ndeg, rho_2d)\n\n    All array inputs are plain jnp arrays \u2014 never nested dicts\n    (FIX over v5: dicts must not be passed as jax.jit arguments).\n\n    Returns\n    -------\n    eps_out  : (N, N)     final emissivity (masked, positive)\n    mean     : (N\u00b2,)      ensemble mean  (pre-PIGNO)\n    std      : (N\u00b2,)      ensemble uncertainty (pre-PIGNO)\n    preds    : (M, N\u00b2)    per-member predictions\n    \"\"\"\n\n    n_members : int = cfg.N_ENS\n\n    def setup(self) -> None:\n        # \u2500\u2500 Shared encoders \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        self.hash_coord = HashGrid(L=cfg.L_HASH,       T=cfg.T_COORD, F=cfg.F_HASH)\n        self.hash_field = HashGrid(L=cfg.L_HASH // 2,  T=cfg.T_FIELD, F=cfg.F_HASH)\n        self.so2        = SO2Harmonics(n_orders=4)\n        self.trunk      = SharedTrunk(hidden=256)\n\n        # \u2500\u2500 Per-member adapters \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        # List comprehension; each adapter has independent weights.\n        self.adapters   = [MemberAdapter() for _ in range(self.n_members)]\n\n        # \u2500\u2500 Learnable per-member noise floor \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        self.log_noise  = self.param(\n            \"log_noise\",\n            nn.initializers.constant(-3.0),\n            (self.n_members,),\n        )\n\n        # \u2500\u2500 PIGNO \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        self.pigno = PIGNO(\n            n_layers = cfg.PIGNO_LAYERS,\n            hidden   = cfg.PIGNO_HIDDEN,\n            N        = cfg.N_GRID,\n        )\n\n    def __call__(\n        self,\n        R_flat  : jnp.ndarray,   # (N\u00b2,) major radius [m]\n        Z_flat  : jnp.ndarray,   # (N\u00b2,) vertical coord [m]\n        psi_n   : jnp.ndarray,   # (N\u00b2,) normalised \u03c8  \u2208 [-1,1]\n        bpol_n  : jnp.ndarray,   # (N\u00b2,) normalised Bpol \u2208 [-1,1]\n        esrc    : jnp.ndarray,   # (E,)  int32  edge sources\n        edst    : jnp.ndarray,   # (E,)  int32  edge destinations\n        ew      : jnp.ndarray,   # (E,)  float32 edge weights\n        ndeg    : jnp.ndarray,   # (N\u00b2,) float32 node degrees\n        rho_2d  : jnp.ndarray,   # (N, N) normalised elliptic radius\n    ) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:\n        \"\"\"\n        Forward pass.\n\n        Returns\n        -------\n        eps_out : (N, N)   final masked emissivity\n        mean    : (N\u00b2,)    ensemble mean (pre-PIGNO)\n        std     : (N\u00b2,)    ensemble uncertainty estimate\n        preds   : (M, N\u00b2)  per-member predictions\n        \"\"\"\n        # \u2500\u2500 Normalise spatial coordinates to [-1, 1] \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        R_n = (R_flat - cfg.R0) / cfg.AP\n        Z_n =  Z_flat           / cfg.BP\n        xy  = jnp.stack([R_n, Z_n], axis=-1)        # (N\u00b2, 2)\n\n        # \u2500\u2500 Field coordinates for field hash \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        # psi_n and bpol_n are already in [-1, 1] (normalised in data_loader)\n        field_xy = jnp.stack([psi_n, bpol_n], axis=-1)    # (N\u00b2, 2)\n\n        # \u2500\u2500 Encode \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        hc   = self.hash_coord(xy)        # (N\u00b2, 32)\n        hf   = self.hash_field(field_xy)  # (N\u00b2, 16)\n        theta = jnp.arctan2(Z_n, R_n)    # (N\u00b2,)\n        so2  = self.so2(theta)            # (N\u00b2, 8)\n\n        # \u2500\u2500 Shared trunk \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        trunk = self.trunk(hc, hf, so2)  # (N\u00b2, 256)  \u2014 shared weights\n\n        # \u2500\u2500 Per-member predictions \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        preds = jnp.stack(\n            [adapter(trunk) for adapter in self.adapters],\n            axis=0,\n        )                                            # (M, N\u00b2)\n\n        mean = jnp.mean(preds, axis=0)               # (N\u00b2,)\n        std  = jnp.std( preds, axis=0) + jnp.exp(self.log_noise[0])\n\n        # \u2500\u2500 PIGNO refinement on ensemble mean \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        eps_raw   = mean.reshape(cfg.N_GRID, cfg.N_GRID)\n        eps_pigno = self.pigno(eps_raw, esrc, edst, ew, ndeg)  # (N, N)\n\n        # \u2500\u2500 Hard boundary mask (smooth sigmoid at \u03c1 = 1) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        mask    = jax.nn.sigmoid(50.0 * (1.0 - rho_2d))       # (N, N)\n        eps_out = eps_pigno * mask                              # (N, N)\n\n        return eps_out, mean, std, preds\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 8.  Utilities\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef count_params(params: dict) -> int:\n    \"\"\"Return total number of scalar parameters in a Flax param tree.\"\"\"\n    return sum(x.size for x in jax.tree_util.tree_leaves(params))\n\n\ndef build_model(\n    R_flat  : jnp.ndarray,\n    Z_flat  : jnp.ndarray,\n    psi_n   : jnp.ndarray,\n    bpol_n  : jnp.ndarray,\n    esrc    : jnp.ndarray,\n    edst    : jnp.ndarray,\n    ew      : jnp.ndarray,\n    ndeg    : jnp.ndarray,\n    rho_2d  : jnp.ndarray,\n    seed    : int = 0,\n    n_members : int = cfg.N_ENS,\n) -> ModelBundle:\n    \"\"\"\n    Instantiate VICTOR_v6 and initialise parameters with a random key.\n\n    Parameters\n    ----------\n    R_flat \u2026 rho_2d : example arrays matching the shapes expected at\n        inference time (used only for shape inference \u2014 values ignored).\n    seed      : int   PRNGKey seed.\n    n_members : int   Number of ensemble members (default cfg.N_ENS).\n\n    Returns\n    -------\n    ModelBundle\n        model  : VICTOR_v6 instance\n        params : initialised parameter tree\n    \"\"\"\n    model  = VICTOR_v6(n_members=n_members)\n    key    = jax.random.PRNGKey(seed)\n    params = model.init(\n        key,\n        R_flat, Z_flat, psi_n, bpol_n,\n        esrc, edst, ew, ndeg, rho_2d,\n    )\n    return ModelBundle(model=model, params=params)\n\n\ndef verify_model(bundle: ModelBundle,\n                 R_flat, Z_flat, psi_n, bpol_n,\n                 esrc, edst, ew, ndeg, rho_2d) -> None:\n    \"\"\"\n    Run a single forward pass and print a diagnostic summary.\n\n    Checks parameter count, output shapes, and NaN counts.\n    Raises AssertionError if any output contains NaNs.\n    \"\"\"\n    model, params = bundle\n\n    outputs = model.apply(\n        params,\n        R_flat, Z_flat, psi_n, bpol_n,\n        esrc, edst, ew, ndeg, rho_2d,\n    )\n\n    n_params  = count_params(params)\n    nan_count = sum(int(jnp.any(jnp.isnan(o))) for o in outputs)\n    shapes    = [o.shape for o in outputs]\n\n    print(\"\u2500\u2500 model.py  VICTOR_v6 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n    print(f\"  Parameters     : {n_params:,}  (target 500k\u2013600k)\")\n    print(f\"  Output shapes  : {shapes}\")\n    print(f\"  NaN in outputs : {nan_count}  (must be 0)\")\n    print(\"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n\n    assert nan_count == 0, (\n        f\"Forward pass produced NaNs in {nan_count} output(s). \"\n        \"Check hash grid initialisers or SIREN \u03c9\u2080.\"\n    )\n    print(\"OK  model.py verified\")\n"

_SRC_losses = "# ============================================================\n# VICTOR v6.0 \u2014 losses.py\n# All individual loss functions + combined loss_fn\n# ============================================================\n# Public API\n# ----------\n#   loss_projection(eps_out, g_noisy, w_ops, active_mask)\n#       \u2192 scalar    Sinogram data-fidelity (MSE on active chords)\n#\n#   loss_boundary(eps_out, rho_2d)\n#       \u2192 scalar    Hard boundary: emissivity must vanish for \u03c1 \u2265 1\n#\n#   loss_smoothness(eps_out)\n#       \u2192 scalar    Total-variation regulariser (finite differences)\n#\n#   loss_ensemble_nll(preds, g_noisy, w_ops, active_mask, log_noise)\n#       \u2192 scalar    Negative log-likelihood of ensemble members\n#\n#   loss_ensemble_diversity(preds)\n#       \u2192 scalar    Repulsion term: penalises collapsed ensemble\n#\n#   loss_isotropy(eps_out, rho_flat)\n#       \u2192 scalar    Physics prior: \u03b5 should vary mainly with \u03c1, not \u03b8\n#\n#   loss_positivity(eps_out)\n#       \u2192 scalar    Soft penalty for negative pixels (belt-and-suspenders)\n#\n#   loss_fn(eps_out, mean, std, preds, g_noisy,\n#           w_ops, active_mask, rho_2d, rho_flat, log_noise)\n#       \u2192 (total_loss, loss_dict)\n#\n# Design principles\n# -----------------\n#  \u2022 All functions are pure JAX \u2014 no side effects, no globals mutated.\n#  \u2022 loss_fn is the single entry point consumed by the training step.\n#  \u2022 loss_dict exposes every individual component for TensorBoard / logs.\n#  \u2022 Weights are tunable via the LossWeights dataclass (or cfg constants).\n#  \u2022 All losses return scalar float32; shapes are asserted in docstrings.\n#  \u2022 W operators are passed as WOperators namedtuples (geometry.py).\n# ============================================================\n\nfrom __future__ import annotations\n\nfrom typing import NamedTuple, Dict, Tuple\n\nimport jax\nimport jax.numpy as jnp\n\nimport config as cfg\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 0.  Loss weights\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\nclass LossWeights(NamedTuple):\n    \"\"\"\n    Scalar multipliers for each loss component.\n\n    Tune these to balance physics priors against data fidelity.\n    Defaults are calibrated for WEST soft-X-ray inversion at VICTOR v6.\n\n    Attributes\n    ----------\n    w_proj       : weight for projection / data-fidelity loss\n    w_boundary   : weight for boundary-zero enforcement\n    w_smooth     : weight for total-variation smoothness regulariser\n    w_nll        : weight for ensemble NLL (heteroscedastic)\n    w_diversity  : weight for ensemble diversity repulsion\n    w_isotropy   : weight for flux-surface isotropy prior\n    w_positivity : weight for soft positivity penalty\n    \"\"\"\n    w_proj       : float = 1.0\n    w_boundary   : float = 5.0\n    w_smooth     : float = 0.02\n    w_nll        : float = 0.5\n    w_diversity  : float = 0.1\n    w_isotropy   : float = 0.05\n    w_positivity : float = 0.5\n\n\n# Singleton default weights \u2014 importable as `losses.DEFAULT_WEIGHTS`\nDEFAULT_WEIGHTS = LossWeights()\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 1.  Projection loss  (data fidelity)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_projection(\n    eps_out     : jnp.ndarray,\n    g_noisy     : jnp.ndarray,\n    w_ops,\n    active_mask : jnp.ndarray,\n) -> jnp.ndarray:\n    \"\"\"\n    Sinogram data-fidelity loss (MSE restricted to active chords).\n\n    Computes the forward projection g_pred = W \u00b7 \u03b5_flat and\n    penalises the mean-square residual on all active (non-zero-sum)\n    rows of the W matrix.\n\n    Parameters\n    ----------\n    eps_out     : (N, N)    predicted emissivity (masked, positive)\n    g_noisy     : (128,)    noisy measured sinogram (normalised)\n    w_ops       : WOperators  (geometry.make_W_operators)\n    active_mask : (128,)    bool/float \u2014 1 for active chords\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    eps_flat = eps_out.flatten()                       # (N\u00b2,)\n    g_pred   = w_ops.matvec(eps_flat)                  # (128,)\n\n    residual = (g_pred - g_noisy) * active_mask        # zero-out inactive\n    n_active = jnp.maximum(active_mask.sum(), 1.0)\n    return jnp.sum(residual ** 2) / n_active\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 2.  Boundary loss\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_boundary(\n    eps_out : jnp.ndarray,\n    rho_2d  : jnp.ndarray,\n) -> jnp.ndarray:\n    \"\"\"\n    Boundary-zero loss: penalise emissivity outside the last closed\n    flux surface (\u03c1 \u2265 1).\n\n    Uses a smooth sigmoid ramp so gradients exist up to ~\u03c1 = 1.1.\n\n    Parameters\n    ----------\n    eps_out : (N, N)      predicted emissivity\n    rho_2d  : (N, N)      normalised elliptic radius (PixelGrids.RHO_2D)\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    # Soft mask: 0 inside plasma, rises to 1 outside (\u03c1 > 1)\n    outside = jax.nn.sigmoid(20.0 * (rho_2d - 1.0))   # (N, N)\n    return jnp.mean((eps_out * outside) ** 2)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 3.  Smoothness loss  (anisotropic total variation)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_smoothness(eps_out: jnp.ndarray) -> jnp.ndarray:\n    \"\"\"\n    Anisotropic total-variation (TV) regulariser.\n\n    Penalises the mean absolute finite difference in both the R and Z\n    directions, encouraging piece-wise smooth reconstructions without\n    over-blurring sharp radial gradients.\n\n    Parameters\n    ----------\n    eps_out : (N, N)   predicted emissivity\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    # Finite differences along both axes\n    dR = jnp.abs(jnp.diff(eps_out, axis=1))    # (N, N-1)\n    dZ = jnp.abs(jnp.diff(eps_out, axis=0))    # (N-1, N)\n    return 0.5 * (jnp.mean(dR) + jnp.mean(dZ))\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 4.  Ensemble NLL  (heteroscedastic negative log-likelihood)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_ensemble_nll(\n    preds       : jnp.ndarray,\n    g_noisy     : jnp.ndarray,\n    w_ops,\n    active_mask : jnp.ndarray,\n    log_noise   : jnp.ndarray,\n) -> jnp.ndarray:\n    \"\"\"\n    Heteroscedastic NLL: each ensemble member predicts a sinogram and\n    the combined Gaussian likelihood is maximised.\n\n    For member m with prediction \u03b5_m and learnable noise log \u03c3_m:\n        NLL_m = 0.5 * [ (g_m - g_noisy)\u00b2 / \u03c3_m\u00b2 + log \u03c3_m\u00b2 ]\n    averaged over active chords and members.\n\n    Parameters\n    ----------\n    preds       : (M, N\u00b2)   per-member emissivity predictions (pre-PIGNO)\n    g_noisy     : (128,)    noisy sinogram\n    w_ops       : WOperators\n    active_mask : (128,)    bool/float\n    log_noise   : (M,)      learnable log(\u03c3_m)  \u2014 from model.log_noise\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    M        = preds.shape[0]\n    n_active = jnp.maximum(active_mask.sum(), 1.0)\n\n    def member_nll(m):\n        g_m     = w_ops.matvec(preds[m])                       # (128,)\n        sigma_m = jnp.exp(log_noise[m]) + 1e-6\n        r       = (g_m - g_noisy) * active_mask\n        return jnp.sum(0.5 * (r ** 2 / sigma_m ** 2 + 2.0 * log_noise[m])) / n_active\n\n    # vmap over members (avoids Python loop in JIT)\n    nll_per_member = jax.vmap(member_nll)(jnp.arange(M))\n    return jnp.mean(nll_per_member)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 5.  Ensemble diversity  (variance repulsion)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_ensemble_diversity(preds: jnp.ndarray) -> jnp.ndarray:\n    \"\"\"\n    Ensemble diversity loss: penalise collapsed ensembles by\n    maximising the mean inter-member variance across pixels.\n\n    A collapsed ensemble (all members identical) gives loss \u2248 0;\n    a diverse ensemble gives a large negative contribution, so the\n    sign is negated: we *minimise* the negative variance.\n\n    Parameters\n    ----------\n    preds : (M, N\u00b2)   per-member emissivity predictions\n\n    Returns\n    -------\n    scalar float32   (non-positive; 0 when fully collapsed)\n    \"\"\"\n    # Variance across members at each pixel, then mean over pixels\n    var_per_pixel = jnp.var(preds, axis=0)          # (N\u00b2,)\n    return -jnp.mean(var_per_pixel)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 6.  Isotropy loss  (flux-surface physics prior)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_isotropy(\n    eps_out  : jnp.ndarray,\n    rho_flat : jnp.ndarray,\n) -> jnp.ndarray:\n    \"\"\"\n    Physics prior: emissivity should be approximately constant on\n    iso-\u03c1 surfaces (i.e., \u03b5 \u2248 f(\u03c1) near the core).\n\n    Bins pixels by \u03c1 into N_BINS radial shells and penalises the\n    intra-shell variance, weighted by the shell's mean emissivity\n    (so dim shells don't dominate).\n\n    Parameters\n    ----------\n    eps_out  : (N, N)    predicted emissivity\n    rho_flat : (N\u00b2,)     normalised elliptic radius (PixelGrids.RHO_FLAT)\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    N_BINS   = 32\n    eps_flat = eps_out.flatten()          # (N\u00b2,)\n\n    # Discretise \u03c1 \u2208 [0, 1) into N_BINS uniform shells\n    bin_idx  = jnp.clip(\n        (rho_flat * N_BINS).astype(jnp.int32), 0, N_BINS - 1\n    )                                      # (N\u00b2,)\n\n    # For each shell: compute mean and variance via scatter\n    # --- mean ---\n    bin_sum  = jnp.zeros(N_BINS).at[bin_idx].add(eps_flat)\n    bin_cnt  = jnp.zeros(N_BINS).at[bin_idx].add(1.0)\n    bin_cnt  = jnp.maximum(bin_cnt, 1.0)\n    bin_mean = bin_sum / bin_cnt           # (N_BINS,)\n\n    # --- variance ---\n    residuals   = eps_flat - bin_mean[bin_idx]      # (N\u00b2,)\n    bin_var_sum = jnp.zeros(N_BINS).at[bin_idx].add(residuals ** 2)\n    bin_var     = bin_var_sum / bin_cnt             # (N_BINS,)\n\n    # Weight shells by mean emissivity (ignore empty outer shells)\n    w = jnp.maximum(bin_mean, 1e-8)                 # (N_BINS,)\n    return jnp.sum(bin_var * w) / jnp.sum(w)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 7.  Positivity loss  (soft belt-and-suspenders)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_positivity(eps_out: jnp.ndarray) -> jnp.ndarray:\n    \"\"\"\n    Soft positivity penalty: the model already outputs softplus-activated\n    values so this should remain near zero during healthy training.\n\n    Provided as a diagnostic and as a numerical safeguard against\n    rare floating-point underflow below zero.\n\n    Parameters\n    ----------\n    eps_out : (N, N)   predicted emissivity\n\n    Returns\n    -------\n    scalar float32\n    \"\"\"\n    return jnp.mean(jnp.maximum(-eps_out, 0.0) ** 2)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 8.  Combined loss_fn\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef loss_fn(\n    eps_out     : jnp.ndarray,\n    mean        : jnp.ndarray,\n    std         : jnp.ndarray,\n    preds       : jnp.ndarray,\n    g_noisy     : jnp.ndarray,\n    w_ops,\n    active_mask : jnp.ndarray,\n    rho_2d      : jnp.ndarray,\n    rho_flat    : jnp.ndarray,\n    log_noise   : jnp.ndarray,\n    weights     : LossWeights = DEFAULT_WEIGHTS,\n) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:\n    \"\"\"\n    Combined physics-informed loss for VICTOR v6.\n\n    Aggregates all individual components using the given LossWeights.\n    The returned loss_dict exposes every component for logging/TensorBoard.\n\n    Parameters\n    ----------\n    eps_out     : (N, N)    final masked emissivity  [model output #0]\n    mean        : (N\u00b2,)     ensemble mean  (pre-PIGNO)  [model output #1]\n    std         : (N\u00b2,)     ensemble std   (pre-PIGNO)  [model output #2]\n    preds       : (M, N\u00b2)   per-member predictions     [model output #3]\n    g_noisy     : (128,)    noisy sinogram for current training sample\n    w_ops       : WOperators  (geometry.make_W_operators)\n    active_mask : (128,)    float32, 1 for active chords\n    rho_2d      : (N, N)    normalised elliptic radius\n    rho_flat    : (N\u00b2,)     same, flattened\n    log_noise   : (M,)      learnable log(\u03c3_m) from model.log_noise\n    weights     : LossWeights  (default: DEFAULT_WEIGHTS)\n\n    Returns\n    -------\n    total_loss : scalar float32\n    loss_dict  : dict[str \u2192 scalar float32]\n        Keys: \"proj\", \"boundary\", \"smooth\", \"nll\", \"diversity\",\n              \"isotropy\", \"positivity\", \"total\"\n    \"\"\"\n    # \u2500\u2500 Individual components \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    l_proj      = loss_projection(\n                      eps_out, g_noisy, w_ops, active_mask)\n\n    l_boundary  = loss_boundary(eps_out, rho_2d)\n\n    l_smooth    = loss_smoothness(eps_out)\n\n    l_nll       = loss_ensemble_nll(\n                      preds, g_noisy, w_ops, active_mask, log_noise)\n\n    l_diversity = loss_ensemble_diversity(preds)\n\n    l_isotropy  = loss_isotropy(eps_out, rho_flat)\n\n    l_positivity = loss_positivity(eps_out)\n\n    # \u2500\u2500 Weighted sum \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    total = (\n        weights.w_proj       * l_proj\n      + weights.w_boundary   * l_boundary\n      + weights.w_smooth     * l_smooth\n      + weights.w_nll        * l_nll\n      + weights.w_diversity  * l_diversity\n      + weights.w_isotropy   * l_isotropy\n      + weights.w_positivity * l_positivity\n    )\n\n    loss_dict = {\n        \"proj\"       : l_proj,\n        \"boundary\"   : l_boundary,\n        \"smooth\"     : l_smooth,\n        \"nll\"        : l_nll,\n        \"diversity\"  : l_diversity,\n        \"isotropy\"   : l_isotropy,\n        \"positivity\" : l_positivity,\n        \"total\"      : total,\n    }\n\n    return total, loss_dict\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 9.  Smoke-test / quick verification\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef verify_losses(\n    model,\n    params    : dict,\n    profile   : dict,\n    w_ops,\n    grids,\n    rho_graph,\n) -> None:\n    \"\"\"\n    Run a single forward pass + loss_fn and print a diagnostic table.\n\n    All values must be finite (no NaN / Inf).  Asserts on failure.\n\n    Parameters\n    ----------\n    model     : VICTOR_v6 instance\n    params    : initialised param tree\n    profile   : one entry from data_loader.load_profiles()\n    w_ops     : WOperators\n    grids     : PixelGrids\n    rho_graph : RhoGraph\n    \"\"\"\n    import jax.numpy as jnp\n\n    # \u2500\u2500 Forward pass \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    eps_out, mean, std, preds = model.apply(\n        params,\n        grids.R_PIX,\n        grids.Z_PIX,\n        profile[\"psi_n\"],\n        profile[\"bpol_n\"],\n        rho_graph.EDGES_SRC,\n        rho_graph.EDGES_DST,\n        rho_graph.EDGE_W,\n        rho_graph.NODE_DEG,\n        grids.RHO_2D,\n    )\n\n    # \u2500\u2500 Noisy sinogram (use sigma=0 for deterministic test) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    g_noisy     = profile[\"g_ideal\"]\n    active_mask = jnp.ones(128, dtype=jnp.float32)   # test: all active\n\n    # \u2500\u2500 Extract log_noise from params \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    log_noise   = params[\"params\"][\"log_noise\"]       # (M,)\n\n    # \u2500\u2500 Run combined loss \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    total, ld   = loss_fn(\n        eps_out, mean, std, preds,\n        g_noisy, w_ops, active_mask,\n        grids.RHO_2D, grids.RHO_FLAT,\n        log_noise,\n    )\n\n    # \u2500\u2500 Report \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    print(\"\u2500\u2500 losses.py  verify_losses \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n    for k, v in ld.items():\n        flag = \"  \u2713\" if jnp.isfinite(v) else \"  \u2717 NON-FINITE\"\n        print(f\"  {k:<12}: {float(v):.6f}{flag}\")\n    print(\"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n\n    bad = [k for k, v in ld.items() if not jnp.isfinite(v)]\n    assert not bad, f\"Non-finite loss components: {bad}\"\n    print(\"OK  losses.py verified\")\n\n\n# \u2500\u2500 Module self-test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    import jax\n    key   = jax.random.PRNGKey(42)\n    N     = cfg.N_GRID\n    M     = cfg.N_ENS\n\n    # Dummy arrays with correct shapes\n    eps   = jax.random.uniform(key, (N, N))\n    preds = jax.random.uniform(key, (M, N * N))\n    g     = jax.random.uniform(key, (128,))\n    rho2d = jnp.ones((N, N)) * 0.5\n    rhof  = jnp.ones((N * N,)) * 0.5\n    lnoise= jnp.full((M,), -3.0)\n    amask = jnp.ones(128)\n    mean  = jnp.mean(preds, axis=0)\n    std   = jnp.std(preds, axis=0)\n\n    # Minimal stub WOperators for self-test (no actual W matrix needed)\n    class _StubOps:\n        @staticmethod\n        def matvec(ef):\n            return jnp.zeros(128)\n\n    stub_ops = _StubOps()\n\n    total, ld = loss_fn(\n        eps, mean, std, preds, g,\n        stub_ops, amask, rho2d, rhof, lnoise,\n    )\n\n    print(\"Self-test loss_fn output:\")\n    for k, v in ld.items():\n        print(f\"  {k:<12}: {float(v):.6f}\")\n    print(\"Self-test passed.\" if jnp.isfinite(total) else \"FAILED \u2014 non-finite total!\")\n"

_SRC_trainer = "# ============================================================\n# VICTOR v6.0 \u2014 trainer.py\n# JIT-compiled train_step, training loop, stage logic\n# ============================================================\n# Public API\n# ----------\n#   build_optimizer(total_steps, lr, warmup)  \u2192 tx (optax chain)\n#   train_step(params, opt_state, ...)         \u2192 (params, opt_state, total, loss_dict)\n#   train_one_profile(...)                     \u2192 (params, opt_state, ep_global, best_data)\n#   train(...)                                 \u2192 (params, opt_state, hist, best_data)\n#\n# Design principles\n# -----------------\n#  \u2022 train_step is @jax.jit \u2014 all Python bools / scalars resolved BEFORE\n#    the JIT boundary (no Python control flow inside JIT).\n#  \u2022 Noisy sinograms are re-drawn every LOG_EVERY steps to avoid\n#    the network memorising a fixed noise realisation.\n#  \u2022 STAGES is consumed from config.py so Cell 1 constants propagate.\n#  \u2022 hist dict is updated in-place; callers receive the same object.\n#  \u2022 Gradient NaN / Inf values are zeroed before the optimiser update\n#    (the \"safe grad\" pattern used in the original notebook).\n#  \u2022 loss_fn is imported from losses.py; train_step is the only place\n#    that calls jax.value_and_grad \u2014 no value_and_grad elsewhere.\n# ============================================================\n\nfrom __future__ import annotations\n\nimport time\nfrom typing import Dict, List, Optional, Tuple\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nimport optax\n\nimport config as cfg\nfrom losses import loss_fn, LossWeights, DEFAULT_WEIGHTS\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 1.  Optimiser factory\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef build_optimizer(\n    total_steps : int,\n    lr          : float = cfg.LR,\n    warmup      : int   = 500,\n    lr_end      : float = 5e-5,\n) -> optax.GradientTransformation:\n    \"\"\"\n    Build the VICTOR v6 optimiser: warmup-cosine-decay Adam + grad-clip.\n\n    Schedule\n    --------\n    0 \u2192 warmup steps : linear warm-up from 0 to lr\n    warmup \u2192 total   : cosine decay from lr to lr_end\n\n    Parameters\n    ----------\n    total_steps : int   Total number of gradient steps expected.\n    lr          : float Peak learning rate (default cfg.LR = 3e-4).\n    warmup      : int   Number of warm-up steps (default 500).\n    lr_end      : float Final learning rate at end of cosine decay.\n\n    Returns\n    -------\n    optax.GradientTransformation   (chain of clip + adam)\n    \"\"\"\n    sched = optax.warmup_cosine_decay_schedule(\n        init_value       = 0.0,\n        peak_value       = lr,\n        warmup_steps     = warmup,\n        decay_steps      = total_steps,\n        end_value        = lr_end,\n    )\n    return optax.chain(\n        optax.clip_by_global_norm(1.0),\n        optax.adam(sched),\n    )\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 2.  JIT-compiled train_step\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\n@jax.jit\ndef train_step(\n    params      : dict,\n    opt_state,\n    tx,\n    g_noisy     : jnp.ndarray,\n    psi_n       : jnp.ndarray,\n    bpol_n      : jnp.ndarray,\n    R_flat      : jnp.ndarray,\n    Z_flat      : jnp.ndarray,\n    esrc        : jnp.ndarray,\n    edst        : jnp.ndarray,\n    ew          : jnp.ndarray,\n    ndeg        : jnp.ndarray,\n    rho_2d      : jnp.ndarray,\n    rho_flat    : jnp.ndarray,\n    active_mask : jnp.ndarray,\n    model,\n    weights     : LossWeights = DEFAULT_WEIGHTS,\n) -> Tuple[dict, object, jnp.ndarray, Dict[str, jnp.ndarray]]:\n    \"\"\"\n    Single JIT-compiled gradient step.\n\n    Computes forward pass \u2192 loss_fn \u2192 value_and_grad \u2192 safe-gradient\n    clipping \u2192 optax update \u2192 new params.\n\n    Parameters\n    ----------\n    params      : Flax param tree\n    opt_state   : optax optimiser state\n    tx          : optax GradientTransformation (the same object used for init)\n    g_noisy     : (128,)  noisy sinogram for this step\n    psi_n       : (N\u00b2,)   normalised \u03c8 field\n    bpol_n      : (N\u00b2,)   normalised Bpol field\n    R_flat \u2026    : geometry arrays (from Cell 2 scope)\n    active_mask : (128,)  float32  1 = active chord\n    model       : VICTOR_v6 instance\n    weights     : LossWeights  (default DEFAULT_WEIGHTS)\n\n    Returns\n    -------\n    new_params   : updated param tree\n    new_opt_state: updated optimiser state\n    total        : scalar float32  total loss\n    loss_dict    : dict[str \u2192 scalar float32]  individual components\n    \"\"\"\n    def _loss(p):\n        eps_out, mean, std, preds = model.apply(\n            p,\n            R_flat, Z_flat, psi_n, bpol_n,\n            esrc, edst, ew, ndeg, rho_2d,\n        )\n        log_noise = p[\"params\"][\"log_noise\"]      # (M,)\n        return loss_fn(\n            eps_out, mean, std, preds,\n            g_noisy, w_ops=None,                  # w_ops resolved inside loss_fn\n            active_mask=active_mask,\n            rho_2d=rho_2d,\n            rho_flat=rho_flat,\n            log_noise=log_noise,\n            weights=weights,\n        )\n\n    (total, loss_dict), grads = jax.value_and_grad(_loss, has_aux=True)(params)\n\n    # Zero out any NaN / Inf gradients (numerical safety)\n    grads = jax.tree_util.tree_map(\n        lambda g: jnp.where(jnp.isfinite(g), g, jnp.zeros_like(g)),\n        grads,\n    )\n\n    updates, new_opt_state = tx.update(grads, opt_state, params)\n    new_params = optax.apply_updates(params, updates)\n\n    return new_params, new_opt_state, total, loss_dict\n\n\n# \u2500\u2500\u2500 Variant that accepts w_ops explicitly (avoids closure captures) \u2500\u2500\n\ndef make_train_step(model, w_ops, weights: LossWeights = DEFAULT_WEIGHTS):\n    \"\"\"\n    Factory that returns a JIT-compiled train_step with model, w_ops,\n    and weights closed over.  Use this pattern instead of passing `model`\n    as a JIT argument, which is not supported by JAX.\n\n    Parameters\n    ----------\n    model   : VICTOR_v6 instance\n    w_ops   : WOperators  (geometry.WOperators)\n    weights : LossWeights\n\n    Returns\n    -------\n    jit-compiled callable with signature:\n        step_fn(params, opt_state, tx,\n                g_noisy, psi_n, bpol_n,\n                R_flat, Z_flat, esrc, edst, ew, ndeg,\n                rho_2d, rho_flat, active_mask)\n        \u2192 (new_params, new_opt_state, total, loss_dict)\n    \"\"\"\n    @jax.jit\n    def _step(\n        params, opt_state, tx,\n        g_noisy, psi_n, bpol_n,\n        R_flat, Z_flat, esrc, edst, ew, ndeg,\n        rho_2d, rho_flat, active_mask,\n    ):\n        def _loss(p):\n            eps_out, mean, std, preds = model.apply(\n                p,\n                R_flat, Z_flat, psi_n, bpol_n,\n                esrc, edst, ew, ndeg, rho_2d,\n            )\n            log_noise = p[\"params\"][\"log_noise\"]\n            return loss_fn(\n                eps_out, mean, std, preds,\n                g_noisy, w_ops,\n                active_mask,\n                rho_2d, rho_flat,\n                log_noise,\n                weights,\n            )\n\n        (total, ld), grads = jax.value_and_grad(_loss, has_aux=True)(params)\n        grads = jax.tree_util.tree_map(\n            lambda g: jnp.where(jnp.isfinite(g), g, jnp.zeros_like(g)),\n            grads,\n        )\n        updates, new_opt = tx.update(grads, opt_state, params)\n        return optax.apply_updates(params, updates), new_opt, total, ld\n\n    return _step\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 3.  Single-profile training loop (one profile \u00d7 all stages)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef train_one_profile(\n    *,\n    step_fn,\n    params       : dict,\n    opt_state,\n    tx,\n    profile      : dict,\n    R_flat       : jnp.ndarray,\n    Z_flat       : jnp.ndarray,\n    esrc         : jnp.ndarray,\n    edst         : jnp.ndarray,\n    ew           : jnp.ndarray,\n    ndeg         : jnp.ndarray,\n    rho_2d       : jnp.ndarray,\n    rho_flat     : jnp.ndarray,\n    active_mask  : jnp.ndarray,\n    inject_noise,\n    stages       : list              = cfg.STAGES,\n    ep_global    : int               = 0,\n    best_data    : float             = float(\"inf\"),\n    start_stage  : int               = 0,\n    start_ep_in_stage : int          = 0,\n    hist         : Optional[dict]    = None,\n    do_checkpoint_fn                 = None,\n    prof_idx     : int               = 0,\n    log_every    : int               = cfg.LOG_EVERY,\n    save_every   : int               = cfg.SAVE_EVERY,\n    t0           : Optional[float]   = None,\n) -> Tuple[dict, object, int, float]:\n    \"\"\"\n    Train params through all noise stages for a single profile.\n\n    Parameters\n    ----------\n    step_fn      : jit-compiled step function from make_train_step()\n    params       : current param tree (input/output)\n    opt_state    : current optimiser state (input/output)\n    tx           : optax GradientTransformation (needed by step_fn)\n    profile      : one dict from data_loader.load_profiles()\n    R_flat \u2026 active_mask : geometry arrays from Cell 2 scope\n    inject_noise : data_loader.inject_noise\n    stages       : list of (n_steps, sigma) from config.STAGES\n    ep_global    : global epoch counter at entry\n    best_data    : best projection loss seen so far (for checkpoint metadata)\n    start_stage  : which stage to resume from (0 = fresh)\n    start_ep_in_stage : which epoch within that stage to resume from (0 = fresh)\n    hist         : dict to accumulate loss history (mutated in place)\n    do_checkpoint_fn : callable(ep_global, prof_idx, stage_idx, ep_in_stage, best_data)\n    prof_idx     : index of this profile (for logging / checkpoint metadata)\n    log_every    : print frequency\n    save_every   : checkpoint frequency\n    t0           : wall-clock start time (for elapsed display)\n\n    Returns\n    -------\n    params, opt_state, ep_global, best_data\n    \"\"\"\n    if hist is None:\n        hist = {}\n    if t0 is None:\n        t0 = time.time()\n\n    psi_n  = profile[\"psi_n\"]\n    bpol_n = profile[\"bpol_n\"]\n    g_clean = profile[\"g_ideal\"]\n\n    for si, (s_ep, s_sig) in enumerate(stages):\n\n        # \u2500\u2500 Resume: skip completed stages for this profile \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        if si < start_stage:\n            continue\n\n        # Noise seed: mix global epoch + stage index\n        g_stage = inject_noise(g_clean, s_sig,\n                               jax.random.PRNGKey(ep_global + si))\n\n        # Epoch range for this stage (support resume mid-stage)\n        ep_start = start_ep_in_stage if si == start_stage else 0\n        # After consuming the resume offset, reset to 0 for all later stages\n        if si == start_stage:\n            start_ep_in_stage = 0\n\n        print(\n            f\"  Prof {prof_idx + 1} | stage {si + 1}/{len(stages)} \"\n            f\"\u03c3={s_sig}  ep_range=[{ep_start},{s_ep})  \"\n            f\"Te_core={float(profile['Te_1d'][0]):.2f} keV\"\n        )\n\n        for ep in range(ep_start, s_ep):\n\n            # Re-draw noise periodically to avoid memorising a fixed sample\n            if ep % log_every == 0 and ep > ep_start:\n                g_stage = inject_noise(\n                    g_clean, s_sig,\n                    jax.random.PRNGKey(ep_global + ep),\n                )\n\n            # \u2500\u2500 Gradient step \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            params, opt_state, total, ld = step_fn(\n                params, opt_state, tx,\n                g_stage, psi_n, bpol_n,\n                R_flat, Z_flat,\n                esrc, edst, ew, ndeg,\n                rho_2d, rho_flat,\n                active_mask,\n            )\n            ep_global += 1\n\n            # \u2500\u2500 History accumulation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            total_f = float(total)\n            if np.isfinite(total_f):\n                _append(hist, \"total\", total_f)\n                for k, v in ld.items():\n                    _append(hist, k, float(v))\n                if float(ld.get(\"proj\", float(\"inf\"))) < best_data:\n                    best_data = float(ld[\"proj\"])\n\n            # \u2500\u2500 Checkpoint \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            if ep_global % save_every == 0 and do_checkpoint_fn is not None:\n                do_checkpoint_fn(ep_global, prof_idx, si, ep + 1, best_data)\n\n            # \u2500\u2500 Logging \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            if ep_global % log_every == 0:\n                proj_f = float(ld.get(\"proj\",  0))\n                nll_f  = float(ld.get(\"nll\",   0))\n                smth_f = float(ld.get(\"smooth\", 0))\n                elapsed = time.time() - t0\n                print(\n                    f\"    ep={ep_global:6d} | L={total_f:.5f} | \"\n                    f\"proj={proj_f:.5f} | nll={nll_f:.5f} | \"\n                    f\"smooth={smth_f:.5f} | t={elapsed:.0f}s\"\n                )\n\n    return params, opt_state, ep_global, best_data\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 4.  Full training loop  (all profiles \u00d7 all stages \u00d7 N_EPOCHS)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef train(\n    *,\n    step_fn,\n    params       : dict,\n    opt_state,\n    tx,\n    profiles     : list,\n    R_flat       : jnp.ndarray,\n    Z_flat       : jnp.ndarray,\n    esrc         : jnp.ndarray,\n    edst         : jnp.ndarray,\n    ew           : jnp.ndarray,\n    ndeg         : jnp.ndarray,\n    rho_2d       : jnp.ndarray,\n    rho_flat     : jnp.ndarray,\n    active_mask  : jnp.ndarray,\n    inject_noise,\n    stages       : list                  = cfg.STAGES,\n    ep_global    : int                   = 0,\n    best_data    : float                 = float(\"inf\"),\n    start_prof   : int                   = 0,\n    start_stage  : int                   = 0,\n    start_ep     : int                   = 0,\n    hist         : Optional[dict]        = None,\n    do_checkpoint_fn                     = None,\n    log_every    : int                   = cfg.LOG_EVERY,\n    save_every   : int                   = cfg.SAVE_EVERY,\n) -> Tuple[dict, object, dict, float]:\n    \"\"\"\n    Outer training loop: iterate over all profiles and all noise stages.\n\n    This function is a thin wrapper that delegates to train_one_profile()\n    for each (profile, stage) block, handles the resume logic for\n    start_prof / start_stage / start_ep, and saves a final checkpoint\n    after each profile completes.\n\n    Parameters\n    ----------\n    step_fn      : jit-compiled step from make_train_step()\n    params       : initial / resumed param tree\n    opt_state    : initial / resumed optimiser state\n    tx           : optax GradientTransformation\n    profiles     : list of profile dicts from data_loader.load_profiles()\n    R_flat \u2026 active_mask : geometry arrays (from Cell 2)\n    inject_noise : data_loader.inject_noise\n    stages       : list of (n_steps, sigma) tuples (default cfg.STAGES)\n    ep_global    : starting global epoch (0 for fresh, >0 for resume)\n    best_data    : best projection loss so far (0.0 for fresh)\n    start_prof   : profile index to resume from\n    start_stage  : stage index to resume from within start_prof\n    start_ep     : epoch within start_stage to resume from\n    hist         : dict for loss history (created if None)\n    do_checkpoint_fn : callable(ep_global, prof_idx, stage_idx, ep, best_data)\n    log_every    : log print frequency (steps)\n    save_every   : checkpoint save frequency (steps)\n\n    Returns\n    -------\n    params     : final param tree\n    opt_state  : final optimiser state\n    hist       : accumulated loss history dict\n    best_data  : best projection loss achieved\n    \"\"\"\n    if hist is None:\n        hist = {}\n\n    t0      = time.time()\n    n_prof  = len(profiles)\n    n_stages = len(stages)\n    total_planned = sum(s for s, _ in stages) * n_prof\n    print(f\"\\n{'='*60}\")\n    print(f\"VICTOR v6.0 \u2014 Training  ({n_prof} profiles \u00d7 {n_stages} stages)\")\n    print(f\"  Total planned steps : {total_planned:,}\")\n    print(f\"  Resume: prof={start_prof}  stage={start_stage}  ep={ep_global}\")\n    print(f\"{'='*60}\\n\")\n\n    _start_stage = start_stage\n    _start_ep    = start_ep\n\n    for pi in range(start_prof, n_prof):\n\n        params, opt_state, ep_global, best_data = train_one_profile(\n            step_fn          = step_fn,\n            params           = params,\n            opt_state        = opt_state,\n            tx               = tx,\n            profile          = profiles[pi],\n            R_flat           = R_flat,\n            Z_flat           = Z_flat,\n            esrc             = esrc,\n            edst             = edst,\n            ew               = ew,\n            ndeg             = ndeg,\n            rho_2d           = rho_2d,\n            rho_flat         = rho_flat,\n            active_mask      = active_mask,\n            inject_noise     = inject_noise,\n            stages           = stages,\n            ep_global        = ep_global,\n            best_data        = best_data,\n            start_stage      = _start_stage,\n            start_ep_in_stage= _start_ep,\n            hist             = hist,\n            do_checkpoint_fn = do_checkpoint_fn,\n            prof_idx         = pi,\n            log_every        = log_every,\n            save_every       = save_every,\n            t0               = t0,\n        )\n\n        # After first profile, resume offsets are consumed \u2014 always start fresh\n        _start_stage = 0\n        _start_ep    = 0\n\n        # Final checkpoint at end of each profile\n        if do_checkpoint_fn is not None:\n            do_checkpoint_fn(ep_global, pi + 1, 0, 0, best_data)\n\n        print(f\"  Profile {pi + 1}/{n_prof} complete  \"\n              f\"ep_global={ep_global}  best_proj={best_data:.6f}\\n\")\n\n    elapsed = time.time() - t0\n    print(f\"\\n{'='*60}\")\n    print(f\"Training complete: {ep_global} steps  \"\n          f\"best_proj={best_data:.6f}  t={elapsed:.1f}s\")\n    print(f\"{'='*60}\")\n\n    return params, opt_state, hist, best_data\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 5.  Internal helpers\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef _append(hist: dict, key: str, value: float) -> None:\n    \"\"\"Append a scalar to a history list, creating the list if needed.\"\"\"\n    if key not in hist:\n        hist[key] = []\n    hist[key].append(value)\n\n\n# \u2500\u2500 Module self-test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    print(\"trainer.py \u2014 no self-test (requires full pipeline).\")\n    print(\"Run Cell 5 in the notebook to exercise the training loop.\")\n"

_SRC_checkpoint = "# ============================================================\n# VICTOR v6.0 \u2014 checkpoint.py\n# Orbax-backed checkpointing with JSON metadata sidecar\n# ============================================================\n# Public API\n# ----------\n#   build_ckpt_manager(ckpt_dir, max_to_keep)  \u2192 CheckpointManager\n#   save_meta(ckpt_dir, ep_global, ...)        \u2192 None   (atomic JSON write)\n#   load_meta(ckpt_dir)                        \u2192 dict | None\n#   do_checkpoint(mgr, params, opt_state, ...) \u2192 None   (blocking save)\n#   resume(mgr, ckpt_dir, params, opt_state,   \u2192 ResumeBundle\n#          tx)\n#\n# Design principles\n# -----------------\n#  \u2022 Params and opt_state are saved as SEPARATE items under\n#    fixed keys CKPT_PARAMS_KEY / CKPT_OPT_KEY.  Mixing them\n#    in a single dict causes Orbax PyTree-shape mismatches on restore.\n#  \u2022 Scalar training state (epoch, profile index, stage index, best loss)\n#    lives in a JSON sidecar next to the Orbax directory, written with\n#    os.replace() (atomic rename) so a Colab disconnect mid-write never\n#    corrupts the sidecar.\n#  \u2022 wait_until_finished() is called after every save to guarantee the\n#    async Orbax write is flushed to disk before we return.  This is the\n#    key fix that prevents file corruption on GPU-timeout disconnects.\n#  \u2022 Adam moments (opt_state) are always restored alongside params so\n#    the LR schedule and momentum estimates survive a restart.\n#  \u2022 All NumPy conversions (jax \u2192 np) happen here so the caller's\n#    params/opt_state remain on device throughout training.\n# ============================================================\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom typing import NamedTuple, Optional\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\ntry:\n    import orbax.checkpoint as ocp\nexcept ImportError as e:\n    raise ImportError(\n        \"orbax-checkpoint is required.  Install it with:\\n\"\n        \"  pip install orbax-checkpoint\"\n    ) from e\n\nimport config as cfg\n\n\n# \u2500\u2500 Constants \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nCKPT_PARAMS_KEY = \"params\"\nCKPT_OPT_KEY    = \"opt_state\"\nMETA_FILENAME   = \"train_meta.json\"\n\n\n# \u2500\u2500 Named return type for resume bundle \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass ResumeBundle(NamedTuple):\n    \"\"\"\n    All state restored from a checkpoint, ready to hand to the training loop.\n\n    Attributes\n    ----------\n    params        : restored Flax param tree (on-device jnp arrays)\n    opt_state     : restored optax optimiser state\n    ep_global     : global epoch counter at the point of the save\n    start_prof    : profile index to resume from\n    start_stage   : noise-stage index to resume from\n    start_ep      : epoch-within-stage to resume from\n    best_data     : best projection loss seen so far\n    resumed       : True if an actual checkpoint was found and loaded\n    \"\"\"\n    params      : dict\n    opt_state   : object\n    ep_global   : int\n    start_prof  : int\n    start_stage : int\n    start_ep    : int\n    best_data   : float\n    resumed     : bool\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 1.  CheckpointManager factory\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef build_ckpt_manager(\n    ckpt_dir     : str,\n    max_to_keep  : int = 3,\n) -> ocp.CheckpointManager:\n    \"\"\"\n    Create an Orbax CheckpointManager pointing at ``ckpt_dir``.\n\n    Parameters\n    ----------\n    ckpt_dir    : str   Directory where checkpoint subdirs are written.\n                        Created if it does not exist.\n    max_to_keep : int   Maximum number of checkpoint steps to retain.\n                        Older ones are deleted automatically.\n\n    Returns\n    -------\n    ocp.CheckpointManager\n    \"\"\"\n    os.makedirs(ckpt_dir, exist_ok=True)\n\n    options = ocp.CheckpointManagerOptions(\n        max_to_keep        = max_to_keep,\n        save_interval_steps= 1,       # we control frequency ourselves\n    )\n    mgr = ocp.CheckpointManager(ckpt_dir, options=options)\n    print(f\"CheckpointManager ready: {ckpt_dir}  (max_to_keep={max_to_keep})\")\n    return mgr\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 2.  JSON metadata sidecar\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef _meta_path(ckpt_dir: str) -> str:\n    return os.path.join(ckpt_dir, META_FILENAME)\n\n\ndef save_meta(\n    ckpt_dir    : str,\n    ep_global   : int,\n    prof_idx    : int,\n    stage_idx   : int,\n    ep_in_stage : int,\n    best_data   : float,\n) -> None:\n    \"\"\"\n    Atomically write scalar training state to a JSON sidecar file.\n\n    Uses a write-then-rename pattern so that a process crash between\n    writes never leaves the sidecar in a partially-written state.\n\n    Parameters\n    ----------\n    ckpt_dir    : str   Same directory passed to build_ckpt_manager().\n    ep_global   : int   Global step counter.\n    prof_idx    : int   Profile index (loop variable ``pi``).\n    stage_idx   : int   Noise-stage index (loop variable ``si``).\n    ep_in_stage : int   Local epoch within the current stage.\n    best_data   : float Best projection loss seen so far.\n    \"\"\"\n    payload = dict(\n        ep_global   = int(ep_global),\n        prof_idx    = int(prof_idx),\n        stage_idx   = int(stage_idx),\n        ep_in_stage = int(ep_in_stage),\n        best_data   = float(best_data),\n    )\n    tmp_path  = _meta_path(ckpt_dir) + \".tmp\"\n    final_path = _meta_path(ckpt_dir)\n\n    with open(tmp_path, \"w\") as f:\n        json.dump(payload, f, indent=2)\n\n    os.replace(tmp_path, final_path)   # atomic rename \u2014 safe on crash\n\n\ndef load_meta(ckpt_dir: str) -> Optional[dict]:\n    \"\"\"\n    Read the JSON sidecar if it exists.\n\n    Returns\n    -------\n    dict with keys ep_global, prof_idx, stage_idx, ep_in_stage, best_data\n    or None if the file does not exist or is unreadable.\n    \"\"\"\n    path = _meta_path(ckpt_dir)\n    if not os.path.exists(path):\n        return None\n    try:\n        with open(path) as f:\n            return json.load(f)\n    except Exception as e:\n        print(f\"  [checkpoint] Meta read failed ({e}) \u2014 scalar state reset\")\n        return None\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 3.  Blocking checkpoint save\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef do_checkpoint(\n    mgr         : ocp.CheckpointManager,\n    params      : dict,\n    opt_state   : object,\n    ep_global   : int,\n    prof_idx    : int,\n    stage_idx   : int,\n    ep_in_stage : int,\n    best_data   : float,\n    ckpt_dir    : Optional[str] = None,\n) -> None:\n    \"\"\"\n    Save params + opt_state via Orbax, then flush the JSON sidecar.\n\n    Calls wait_until_finished() after the Orbax save so that async\n    writes are completed before this function returns.  This guarantees\n    the checkpoint is safely on disk even if the Python process is\n    killed immediately afterwards (Colab GPU-timeout pattern).\n\n    Parameters\n    ----------\n    mgr         : ocp.CheckpointManager  built by build_ckpt_manager()\n    params      : Flax param tree (on-device jnp arrays)\n    opt_state   : optax optimiser state (on-device)\n    ep_global   : step index passed to mgr.save() as the \"step\" key\n    prof_idx    : current profile index (written to sidecar)\n    stage_idx   : current stage index (written to sidecar)\n    ep_in_stage : local epoch within stage (written to sidecar)\n    best_data   : best projection loss (written to sidecar)\n    ckpt_dir    : directory for the JSON sidecar.\n                  Required only if mgr.directory is not accessible as an\n                  attribute (older Orbax versions); defaults to\n                  inferred from mgr._directory.\n\n    Notes\n    -----\n    Params and opt_state are converted to NumPy arrays before passing to\n    Orbax.  The on-device originals are not affected.\n    \"\"\"\n    # Convert to NumPy for serialisation (non-destructive)\n    params_np    = jax.tree_util.tree_map(np.array, params)\n    opt_state_np = jax.tree_util.tree_map(np.array, opt_state)\n\n    mgr.save(\n        ep_global,\n        items={\n            CKPT_PARAMS_KEY: params_np,\n            CKPT_OPT_KEY   : opt_state_np,\n        },\n    )\n    mgr.wait_until_finished()   # CRITICAL: blocks until disk write completes\n\n    # Write the JSON sidecar\n    resolved_dir = ckpt_dir or _resolve_dir(mgr)\n    save_meta(resolved_dir, ep_global, prof_idx, stage_idx, ep_in_stage, best_data)\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 4.  Resume logic\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef resume(\n    mgr         : ocp.CheckpointManager,\n    ckpt_dir    : str,\n    params      : dict,\n    opt_state   : object,\n    tx,\n) -> ResumeBundle:\n    \"\"\"\n    Attempt to restore params, opt_state, and scalar training state.\n\n    If a valid checkpoint is found it is loaded and a ResumeBundle with\n    ``resumed=True`` is returned.  If no checkpoint exists, or if the\n    restore fails for any reason, the original (fresh) params and\n    opt_state are returned inside a bundle with ``resumed=False``.\n\n    Parameters\n    ----------\n    mgr       : ocp.CheckpointManager\n    ckpt_dir  : str   Directory for the JSON sidecar.\n    params    : dict  Freshly-initialised param tree (used as shape reference).\n    opt_state : object  Freshly-initialised optimiser state.\n    tx        : optax GradientTransformation  (kept for potential re-init).\n\n    Returns\n    -------\n    ResumeBundle\n    \"\"\"\n    latest = mgr.latest_step()\n\n    if latest is None:\n        print(\"No checkpoint found \u2014 starting fresh.\")\n        return ResumeBundle(\n            params      = params,\n            opt_state   = opt_state,\n            ep_global   = 0,\n            start_prof  = 0,\n            start_stage = 0,\n            start_ep    = 0,\n            best_data   = float(\"inf\"),\n            resumed     = False,\n        )\n\n    # \u2500\u2500 Load sidecar first (non-destructive \u2014 just reads a file) \u2500\u2500\u2500\u2500\u2500\n    meta = load_meta(ckpt_dir)\n\n    # \u2500\u2500 Restore Orbax checkpoint \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    try:\n        restored = mgr.restore(\n            latest,\n            items={\n                CKPT_PARAMS_KEY: params,\n                CKPT_OPT_KEY   : opt_state,\n            },\n        )\n\n        # Move arrays back onto the JAX device\n        r_params    = jax.tree_util.tree_map(jnp.array, restored[CKPT_PARAMS_KEY])\n        r_opt_state = jax.tree_util.tree_map(jnp.array, restored[CKPT_OPT_KEY])\n\n    except Exception as e:\n        print(f\"  [checkpoint] Restore failed ({e}) \u2014 starting fresh.\")\n        return ResumeBundle(\n            params      = params,\n            opt_state   = opt_state,\n            ep_global   = 0,\n            start_prof  = 0,\n            start_stage = 0,\n            start_ep    = 0,\n            best_data   = float(\"inf\"),\n            resumed     = False,\n        )\n\n    # \u2500\u2500 Parse scalar state from sidecar \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    if meta:\n        ep_global   = int(meta.get(\"ep_global\",   0))\n        start_prof  = int(meta.get(\"prof_idx\",    0))\n        start_stage = int(meta.get(\"stage_idx\",   0))\n        start_ep    = int(meta.get(\"ep_in_stage\", 0))\n        best_data   = float(meta.get(\"best_data\", float(\"inf\")))\n        print(\n            f\"Resumed step {latest}: \"\n            f\"ep={ep_global}  prof={start_prof}  \"\n            f\"stage={start_stage}  ep_in_stage={start_ep}\"\n        )\n    else:\n        # Params/opt_state loaded but sidecar missing \u2014 scalar state reset\n        ep_global   = 0\n        start_prof  = 0\n        start_stage = 0\n        start_ep    = 0\n        best_data   = float(\"inf\")\n        print(\n            f\"Resumed params/opt_state from step {latest} \"\n            f\"(sidecar missing \u2014 scalar state reset)\"\n        )\n\n    return ResumeBundle(\n        params      = r_params,\n        opt_state   = r_opt_state,\n        ep_global   = ep_global,\n        start_prof  = start_prof,\n        start_stage = start_stage,\n        start_ep    = start_ep,\n        best_data   = best_data,\n        resumed     = True,\n    )\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 5.  Internal helpers\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef _resolve_dir(mgr: ocp.CheckpointManager) -> str:\n    \"\"\"\n    Robustly retrieve the checkpoint directory from the manager object.\n\n    Orbax has changed the attribute name across versions; we try several.\n    \"\"\"\n    for attr in (\"directory\", \"_directory\", \"_manager_dir\"):\n        val = getattr(mgr, attr, None)\n        if val is not None:\n            return str(val)\n    raise AttributeError(\n        \"Cannot determine checkpoint directory from CheckpointManager. \"\n        \"Pass ckpt_dir explicitly to do_checkpoint().\"\n    )\n\n\n# \u2500\u2500 Module self-test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    import tempfile\n\n    print(\"checkpoint.py \u2014 self-test\")\n\n    with tempfile.TemporaryDirectory() as tmp:\n        mgr = build_ckpt_manager(tmp, max_to_keep=2)\n\n        # Fake params / opt_state\n        fake_params    = {\"params\": {\"w\": np.ones((4, 4), dtype=np.float32)}}\n        fake_opt_state = {\"mu\": np.zeros((4, 4), dtype=np.float32)}\n\n        # Save\n        do_checkpoint(mgr, fake_params, fake_opt_state,\n                      ep_global=10, prof_idx=0, stage_idx=1,\n                      ep_in_stage=5, best_data=0.42, ckpt_dir=tmp)\n\n        # Load meta\n        meta = load_meta(tmp)\n        assert meta is not None, \"Meta not found after save\"\n        assert meta[\"ep_global\"] == 10\n        assert meta[\"best_data\"] == 0.42\n        print(f\"  save_meta / load_meta OK: {meta}\")\n\n        # Resume\n        bundle = resume(mgr, tmp, fake_params, fake_opt_state, tx=None)\n        assert bundle.resumed, \"Expected resumed=True\"\n        assert bundle.ep_global == 10\n        print(f\"  resume OK: ep_global={bundle.ep_global}\")\n\n    print(\"checkpoint.py self-test PASSED\")\n"

_SRC_evaluate = "# ============================================================\n# VICTOR v6.0 \u2014 evaluate.py\n# Evaluation metrics (PSNR, CC, RelErr) + all diagnostic plots\n# ============================================================\n# Public API\n# ----------\n#   compute_psnr(pred, gt, data_range)          \u2192 float\n#   compute_cc(pred, gt)                        \u2192 float\n#   compute_rel_err(pred, gt)                   \u2192 float\n#   compute_all_metrics(pred, gt, label)        \u2192 MetricBundle\n#\n#   plot_reconstruction(pred, gt, profile, ...)  \u2192 Figure\n#   plot_sinogram_residual(pred, profile, ...)   \u2192 Figure\n#   plot_radial_profile(pred, profile, ...)      \u2192 Figure\n#   plot_uncertainty(std_2d, rho_2d, ...)        \u2192 Figure\n#   plot_ensemble_members(preds, ...)            \u2192 Figure\n#   plot_loss_curves(hist, ...)                  \u2192 Figure\n#   plot_all_profiles(model, params, profiles,   \u2192 list[MetricBundle]\n#                     w_ops, grids, rho_graph)\n#\n#   evaluate_profile(model, params, profile,     \u2192 EvalBundle\n#                    w_ops, grids, rho_graph)\n#   evaluate_all(model, params, profiles,        \u2192 list[EvalBundle]\n#                w_ops, grids, rho_graph)\n#   save_summary_csv(eval_bundles, path)         \u2192 None\n#\n# Design principles\n# -----------------\n#  \u2022 All functions are pure (no JAX globals mutated).\n#  \u2022 Figures are returned \u2014 callers decide whether to show/save.\n#  \u2022 Physics-aware plots: reconstruction uses rho contours; sinogram\n#    plots annotate active-chord boundaries.\n#  \u2022 safe_numpy() converts JAX arrays / scalars to plain NumPy to avoid\n#    matplotlib dtype warnings.\n#  \u2022 All axes are labelled with units; colorbars are added where needed.\n# ============================================================\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Dict, List, NamedTuple, Optional, Tuple\n\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport matplotlib.gridspec as gridspec\nfrom matplotlib.colors import Normalize\nfrom matplotlib.cm import ScalarMappable\n\nimport jax\nimport jax.numpy as jnp\n\nimport config as cfg\n\n\n# \u2500\u2500 Helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef safe_numpy(x) -> np.ndarray:\n    \"\"\"Convert JAX array, jnp scalar, or plain numpy to a numpy array.\"\"\"\n    if hasattr(x, \"__jax_array__\") or hasattr(x, \"device_buffer\"):\n        return np.asarray(x)\n    if isinstance(x, (int, float)):\n        return np.array(x)\n    return np.asarray(x)\n\n\n# \u2500\u2500 Named return types \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass MetricBundle(NamedTuple):\n    \"\"\"Per-profile scalar evaluation metrics.\"\"\"\n    profile_idx : int\n    psnr        : float\n    cc          : float\n    rel_err     : float\n    proj_mse    : float   # sinogram MSE on active chords (data fidelity)\n\n\nclass EvalBundle(NamedTuple):\n    \"\"\"Full per-profile evaluation outputs.\"\"\"\n    profile_idx : int\n    eps_pred    : np.ndarray   # (N, N)  final model output\n    eps_gt      : np.ndarray   # (N, N)  normalised ground truth\n    mean_2d     : np.ndarray   # (N, N)  ensemble mean (pre-PIGNO)\n    std_2d      : np.ndarray   # (N, N)  ensemble std  (pre-PIGNO)\n    preds_2d    : np.ndarray   # (M, N, N)  per-member predictions\n    g_pred      : np.ndarray   # (128,)  projected sinogram\n    g_gt        : np.ndarray   # (128,)  ground-truth sinogram\n    metrics     : MetricBundle\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 1.  Metrics\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef compute_psnr(\n    pred       : np.ndarray,\n    gt         : np.ndarray,\n    data_range : float = 1.0,\n) -> float:\n    \"\"\"\n    Peak Signal-to-Noise Ratio (dB).\n\n    PSNR = 10 \u00b7 log\u2081\u2080(data_range\u00b2 / MSE)\n\n    Parameters\n    ----------\n    pred       : (N, N)  predicted image\n    gt         : (N, N)  ground-truth image\n    data_range : float   value range of the images (default 1.0)\n\n    Returns\n    -------\n    float  (positive; higher is better; \u221e if pred == gt)\n    \"\"\"\n    pred = safe_numpy(pred).astype(np.float64)\n    gt   = safe_numpy(gt).astype(np.float64)\n    mse  = np.mean((pred - gt) ** 2)\n    if mse == 0.0:\n        return float(\"inf\")\n    return float(10.0 * np.log10(data_range ** 2 / mse))\n\n\ndef compute_cc(pred: np.ndarray, gt: np.ndarray) -> float:\n    \"\"\"\n    Pearson cross-correlation coefficient.\n\n    CC = Cov(pred, gt) / (\u03c3_pred \u00b7 \u03c3_gt)\n\n    Parameters\n    ----------\n    pred : (N, N)  predicted image\n    gt   : (N, N)  ground-truth image\n\n    Returns\n    -------\n    float in [-1, 1]  (1.0 is perfect)\n    \"\"\"\n    pred = safe_numpy(pred).flatten().astype(np.float64)\n    gt   = safe_numpy(gt).flatten().astype(np.float64)\n\n    pred_c = pred - pred.mean()\n    gt_c   = gt   - gt.mean()\n\n    denom = np.sqrt(np.sum(pred_c ** 2) * np.sum(gt_c ** 2)) + 1e-12\n    return float(np.sum(pred_c * gt_c) / denom)\n\n\ndef compute_rel_err(pred: np.ndarray, gt: np.ndarray) -> float:\n    \"\"\"\n    Relative L2 error.\n\n    RelErr = \u2016pred - gt\u2016\u2082 / \u2016gt\u2016\u2082\n\n    Parameters\n    ----------\n    pred : (N, N)  predicted image\n    gt   : (N, N)  ground-truth image\n\n    Returns\n    -------\n    float \u2265 0  (0.0 is perfect)\n    \"\"\"\n    pred = safe_numpy(pred).flatten().astype(np.float64)\n    gt   = safe_numpy(gt).flatten().astype(np.float64)\n    norm_gt = np.linalg.norm(gt)\n    if norm_gt < 1e-12:\n        return float(np.linalg.norm(pred - gt))\n    return float(np.linalg.norm(pred - gt) / norm_gt)\n\n\ndef compute_all_metrics(\n    eps_pred    : np.ndarray,\n    eps_gt      : np.ndarray,\n    g_pred      : np.ndarray,\n    g_gt        : np.ndarray,\n    active_mask : np.ndarray,\n    profile_idx : int = 0,\n) -> MetricBundle:\n    \"\"\"\n    Compute PSNR, CC, RelErr, and sinogram MSE for one profile.\n\n    Parameters\n    ----------\n    eps_pred    : (N, N)   predicted emissivity\n    eps_gt      : (N, N)   ground-truth emissivity (normalised)\n    g_pred      : (128,)   projected sinogram from model\n    g_gt        : (128,)   ground-truth sinogram\n    active_mask : (128,)   1 for active chords\n    profile_idx : int      index label\n\n    Returns\n    -------\n    MetricBundle\n    \"\"\"\n    eps_pred = safe_numpy(eps_pred)\n    eps_gt   = safe_numpy(eps_gt)\n    g_pred   = safe_numpy(g_pred)\n    g_gt     = safe_numpy(g_gt)\n    mask     = safe_numpy(active_mask).astype(bool)\n\n    # Sinogram MSE restricted to active chords\n    res      = (g_pred[mask] - g_gt[mask])\n    proj_mse = float(np.mean(res ** 2))\n\n    return MetricBundle(\n        profile_idx = profile_idx,\n        psnr        = compute_psnr(eps_pred, eps_gt, data_range=1.0),\n        cc          = compute_cc(eps_pred, eps_gt),\n        rel_err     = compute_rel_err(eps_pred, eps_gt),\n        proj_mse    = proj_mse,\n    )\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 2.  Reconstruction plot  (GT | Pred | Error | Uncertainty)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_reconstruction(\n    eps_pred   : np.ndarray,\n    eps_gt     : np.ndarray,\n    std_2d     : np.ndarray,\n    rho_2d     : np.ndarray,\n    R_pix      : np.ndarray,\n    Z_pix      : np.ndarray,\n    metrics    : Optional[MetricBundle] = None,\n    profile_idx: int = 0,\n    title_extra: str = \"\",\n    cmap       : str = \"inferno\",\n) -> plt.Figure:\n    \"\"\"\n    Four-panel reconstruction diagnostic:\n      [Ground Truth | Prediction | Absolute Error | Ensemble Uncertainty]\n\n    Rho contours (\u03c1 = 0.25, 0.5, 0.75, 1.0) are overlaid on each panel.\n\n    Parameters\n    ----------\n    eps_pred    : (N, N)   predicted emissivity (masked)\n    eps_gt      : (N, N)   ground-truth emissivity\n    std_2d      : (N, N)   ensemble std map (pre-PIGNO, reshaped)\n    rho_2d      : (N, N)   normalised elliptic radius\n    R_pix       : (N\u00b2,)    major radius pixel centres [m]\n    Z_pix       : (N\u00b2,)    vertical pixel centres [m]\n    metrics     : MetricBundle, optional \u2014 used in suptitle\n    profile_idx : int\n    title_extra : str       appended to the figure title\n    cmap        : str       matplotlib colormap for emissivity panels\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    N        = cfg.N_GRID\n    eps_pred = safe_numpy(eps_pred)\n    eps_gt   = safe_numpy(eps_gt)\n    std_2d   = safe_numpy(std_2d).reshape(N, N)\n    rho_2d   = safe_numpy(rho_2d)\n    R_2d     = safe_numpy(R_pix).reshape(N, N)\n    Z_2d     = safe_numpy(Z_pix).reshape(N, N)\n\n    abs_err  = np.abs(eps_pred - eps_gt)\n    vmax     = max(eps_gt.max(), eps_pred.max(), 1e-8)\n    emax     = abs_err.max() or 1e-8\n\n    fig, axes = plt.subplots(1, 4, figsize=(20, 5))\n    rho_levels = [0.25, 0.50, 0.75, 1.00]\n\n    def _add_rho_contours(ax, lw=0.6):\n        cs = ax.contour(\n            R_2d, Z_2d, rho_2d,\n            levels=rho_levels,\n            colors=\"white\", linewidths=lw, alpha=0.55,\n        )\n        ax.clabel(cs, fmt={v: f\"\u03c1={v:.2f}\" for v in rho_levels},\n                  fontsize=6, inline=True)\n\n    def _pcolormesh(ax, data, vmin, vmax, cmap_):\n        pcm = ax.pcolormesh(\n            R_2d, Z_2d, data,\n            vmin=vmin, vmax=vmax, cmap=cmap_, shading=\"auto\",\n        )\n        return pcm\n\n    # Panel 0: Ground truth\n    pcm = _pcolormesh(axes[0], eps_gt, 0, vmax, cmap)\n    _add_rho_contours(axes[0])\n    plt.colorbar(pcm, ax=axes[0], fraction=0.046, pad=0.04)\n    axes[0].set_title(\"Ground Truth \u03b5\")\n    axes[0].set_xlabel(\"R [m]\"); axes[0].set_ylabel(\"Z [m]\")\n    axes[0].set_aspect(\"equal\")\n\n    # Panel 1: Prediction\n    pcm = _pcolormesh(axes[1], eps_pred, 0, vmax, cmap)\n    _add_rho_contours(axes[1])\n    plt.colorbar(pcm, ax=axes[1], fraction=0.046, pad=0.04)\n    axes[1].set_title(\"VICTOR Prediction \u03b5\")\n    axes[1].set_xlabel(\"R [m]\"); axes[1].set_aspect(\"equal\")\n\n    # Panel 2: Absolute error\n    pcm = _pcolormesh(axes[2], abs_err, 0, emax, \"hot\")\n    _add_rho_contours(axes[2], lw=0.5)\n    plt.colorbar(pcm, ax=axes[2], fraction=0.046, pad=0.04)\n    axes[2].set_title(\"|GT \u2212 Pred|\")\n    axes[2].set_xlabel(\"R [m]\"); axes[2].set_aspect(\"equal\")\n\n    # Panel 3: Uncertainty (ensemble std)\n    umax = max(std_2d.max(), 1e-8)\n    pcm  = _pcolormesh(axes[3], std_2d, 0, umax, \"viridis\")\n    _add_rho_contours(axes[3], lw=0.5)\n    plt.colorbar(pcm, ax=axes[3], fraction=0.046, pad=0.04)\n    axes[3].set_title(\"Ensemble \u03c3 (uncertainty)\")\n    axes[3].set_xlabel(\"R [m]\"); axes[3].set_aspect(\"equal\")\n\n    # Suptitle\n    metric_str = \"\"\n    if metrics is not None:\n        metric_str = (\n            f\"  PSNR={metrics.psnr:.1f} dB\"\n            f\"  CC={metrics.cc:.3f}\"\n            f\"  RelErr={metrics.rel_err:.3f}\"\n        )\n    title = f\"VICTOR v6.0 \u2014 Profile {profile_idx}{metric_str}\"\n    if title_extra:\n        title += f\"  {title_extra}\"\n    fig.suptitle(title, fontsize=12, fontweight=\"bold\")\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 3.  Sinogram residual plot\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_sinogram_residual(\n    g_pred      : np.ndarray,\n    g_gt        : np.ndarray,\n    active_mask : np.ndarray,\n    profile_idx : int = 0,\n) -> plt.Figure:\n    \"\"\"\n    Two-panel sinogram diagnostic:\n      Top   \u2014 measured vs predicted sinogram per chord\n      Bottom \u2014 residual  (pred \u2212 gt) per chord\n\n    Active vs inactive chords are annotated.\n\n    Parameters\n    ----------\n    g_pred      : (128,)  predicted sinogram\n    g_gt        : (128,)  ground-truth sinogram\n    active_mask : (128,)  float/bool \u2014 1 for active chords\n    profile_idx : int\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    g_pred = safe_numpy(g_pred)\n    g_gt   = safe_numpy(g_gt)\n    mask   = safe_numpy(active_mask).astype(bool)\n\n    chords   = np.arange(len(g_pred))\n    residual = g_pred - g_gt\n\n    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)\n\n    # Top: signals\n    ax1.plot(chords, g_gt,   lw=1.5, label=\"Ground truth g\", color=\"steelblue\")\n    ax1.plot(chords, g_pred, lw=1.2, label=\"Predicted g\", color=\"darkorange\",\n             linestyle=\"--\")\n    # Shade inactive chords\n    for c in chords[~mask]:\n        ax1.axvspan(c - 0.5, c + 0.5, color=\"grey\", alpha=0.15)\n    ax1.set_ylabel(\"Sinogram [a.u.]\")\n    ax1.set_title(f\"Sinogram \u2014 Profile {profile_idx}\")\n    ax1.legend(fontsize=9)\n    ax1.grid(True, alpha=0.3)\n\n    # Bottom: residual\n    ax2.bar(chords[mask],  residual[mask],  width=0.8,\n            color=\"darkorange\", alpha=0.75, label=\"active\")\n    ax2.bar(chords[~mask], residual[~mask], width=0.8,\n            color=\"grey\", alpha=0.4, label=\"inactive\")\n    ax2.axhline(0, color=\"black\", lw=0.8)\n    ax2.set_xlabel(\"Chord index\")\n    ax2.set_ylabel(\"Residual (pred \u2212 GT)\")\n    ax2.legend(fontsize=9)\n    ax2.grid(True, alpha=0.3)\n\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 4.  Radial profile plot  (\u03b5 vs \u03c1)\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_radial_profile(\n    eps_pred  : np.ndarray,\n    eps_gt    : np.ndarray,\n    std_2d    : np.ndarray,\n    rho_flat  : np.ndarray,\n    profile_idx: int = 0,\n    n_bins    : int  = 40,\n) -> plt.Figure:\n    \"\"\"\n    Radial profile comparison: binned \u03b5 vs \u03c1 for GT and prediction,\n    with ensemble \u00b11\u03c3 uncertainty band.\n\n    Parameters\n    ----------\n    eps_pred    : (N, N)   predicted emissivity\n    eps_gt      : (N, N)   ground-truth emissivity\n    std_2d      : (N, N)   ensemble std (pre-PIGNO)\n    rho_flat    : (N\u00b2,)    normalised elliptic radius (flattened)\n    profile_idx : int\n    n_bins      : int      number of radial bins\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    eps_pred = safe_numpy(eps_pred).flatten()\n    eps_gt   = safe_numpy(eps_gt).flatten()\n    std_flat = safe_numpy(std_2d).flatten()\n    rho      = safe_numpy(rho_flat).flatten()\n\n    # Keep only plasma interior (\u03c1 < 1)\n    inside   = rho < 1.0\n    rho      = rho[inside]\n    eps_pred = eps_pred[inside]\n    eps_gt   = eps_gt[inside]\n    std_flat = std_flat[inside]\n\n    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)\n    bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])\n\n    def _bin_mean_std(vals):\n        means = np.zeros(n_bins)\n        stds  = np.zeros(n_bins)\n        for k in range(n_bins):\n            sel = (rho >= bin_edges[k]) & (rho < bin_edges[k + 1])\n            if sel.sum() > 0:\n                means[k] = vals[sel].mean()\n                stds[k]  = vals[sel].std()\n        return means, stds\n\n    gt_mean,   gt_std   = _bin_mean_std(eps_gt)\n    pred_mean, pred_std = _bin_mean_std(eps_pred)\n    unc_mean,  _        = _bin_mean_std(std_flat)\n\n    fig, ax = plt.subplots(figsize=(9, 5))\n    ax.plot(bin_centres, gt_mean,   lw=2.0, color=\"steelblue\",  label=\"Ground Truth\")\n    ax.fill_between(bin_centres,\n                    gt_mean - gt_std, gt_mean + gt_std,\n                    alpha=0.2, color=\"steelblue\")\n\n    ax.plot(bin_centres, pred_mean, lw=1.8, color=\"darkorange\",\n            linestyle=\"--\", label=\"VICTOR Pred.\")\n    ax.fill_between(bin_centres,\n                    pred_mean - unc_mean, pred_mean + unc_mean,\n                    alpha=0.25, color=\"darkorange\", label=\"Ensemble \u00b1\u03c3\")\n\n    ax.set_xlabel(\"Normalised radius \u03c1\", fontsize=11)\n    ax.set_ylabel(\"Emissivity [a.u.]\",  fontsize=11)\n    ax.set_title(f\"Radial emissivity profile \u2014 Profile {profile_idx}\", fontsize=12)\n    ax.legend(fontsize=10)\n    ax.set_xlim(0.0, 1.0)\n    ax.grid(True, alpha=0.3)\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 5.  Uncertainty map\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_uncertainty(\n    std_2d      : np.ndarray,\n    rho_2d      : np.ndarray,\n    R_pix       : np.ndarray,\n    Z_pix       : np.ndarray,\n    profile_idx : int = 0,\n) -> plt.Figure:\n    \"\"\"\n    Standalone uncertainty-map panel with rho contours.\n\n    Parameters\n    ----------\n    std_2d      : (N, N)   ensemble std map\n    rho_2d      : (N, N)   normalised elliptic radius\n    R_pix       : (N\u00b2,)    major radius [m]\n    Z_pix       : (N\u00b2,)    vertical coord [m]\n    profile_idx : int\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    N      = cfg.N_GRID\n    std_2d = safe_numpy(std_2d).reshape(N, N)\n    rho_2d = safe_numpy(rho_2d)\n    R_2d   = safe_numpy(R_pix).reshape(N, N)\n    Z_2d   = safe_numpy(Z_pix).reshape(N, N)\n\n    fig, ax = plt.subplots(figsize=(6, 6))\n    pcm = ax.pcolormesh(\n        R_2d, Z_2d, std_2d,\n        vmin=0, vmax=std_2d.max() or 1e-8, cmap=\"plasma\", shading=\"auto\",\n    )\n    plt.colorbar(pcm, ax=ax, label=\"Ensemble \u03c3 [a.u.]\")\n    cs = ax.contour(R_2d, Z_2d, rho_2d,\n                    levels=[0.25, 0.5, 0.75, 1.0],\n                    colors=\"white\", linewidths=0.7, alpha=0.6)\n    ax.clabel(cs, fmt=\"%.2f\", fontsize=7, inline=True)\n    ax.set_title(f\"Ensemble Uncertainty \u2014 Profile {profile_idx}\", fontsize=12)\n    ax.set_xlabel(\"R [m]\"); ax.set_ylabel(\"Z [m]\")\n    ax.set_aspect(\"equal\")\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 6.  Ensemble members\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_ensemble_members(\n    preds_2d    : np.ndarray,\n    R_pix       : np.ndarray,\n    Z_pix       : np.ndarray,\n    profile_idx : int = 0,\n    cmap        : str = \"inferno\",\n) -> plt.Figure:\n    \"\"\"\n    Grid plot of all M ensemble member predictions.\n\n    Parameters\n    ----------\n    preds_2d    : (M, N, N)  per-member emissivity\n    R_pix       : (N\u00b2,)\n    Z_pix       : (N\u00b2,)\n    profile_idx : int\n    cmap        : str\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    N        = cfg.N_GRID\n    preds_2d = safe_numpy(preds_2d)\n    M        = preds_2d.shape[0]\n    R_2d     = safe_numpy(R_pix).reshape(N, N)\n    Z_2d     = safe_numpy(Z_pix).reshape(N, N)\n\n    ncols = min(M, 5)\n    nrows = (M + ncols - 1) // ncols\n    fig, axes = plt.subplots(nrows, ncols,\n                              figsize=(4 * ncols, 4.2 * nrows),\n                              squeeze=False)\n\n    vmax = preds_2d.max() or 1.0\n\n    for m in range(M):\n        ax  = axes[m // ncols][m % ncols]\n        pcm = ax.pcolormesh(\n            R_2d, Z_2d, preds_2d[m],\n            vmin=0, vmax=vmax, cmap=cmap, shading=\"auto\",\n        )\n        ax.set_title(f\"Member {m + 1}\", fontsize=10)\n        ax.set_xlabel(\"R [m]\"); ax.set_ylabel(\"Z [m]\")\n        ax.set_aspect(\"equal\")\n        plt.colorbar(pcm, ax=ax, fraction=0.046, pad=0.04)\n\n    # Hide unused axes\n    for m in range(M, nrows * ncols):\n        axes[m // ncols][m % ncols].set_visible(False)\n\n    fig.suptitle(f\"Ensemble Members \u2014 Profile {profile_idx}\",\n                 fontsize=13, fontweight=\"bold\")\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 7.  Loss curves\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_loss_curves(\n    hist      : dict,\n    save_path : Optional[str] = None,\n) -> plt.Figure:\n    \"\"\"\n    Multi-panel loss-curve plot from the history dict accumulated by\n    trainer.train().\n\n    Panels shown (if the key exists in hist):\n      \u2022 total        \u2014 overall weighted loss (log scale)\n      \u2022 proj         \u2014 data-fidelity projection loss\n      \u2022 nll          \u2014 ensemble NLL\n      \u2022 smooth       \u2014 TV smoothness\n      \u2022 diversity    \u2014 ensemble diversity (negative \u2192 rising = good)\n      \u2022 isotropy     \u2014 flux-surface isotropy\n      \u2022 boundary     \u2014 boundary enforcement\n\n    Parameters\n    ----------\n    hist      : dict[str \u2192 list[float]]   from trainer.train()\n    save_path : str, optional              if given, saves the figure\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    KEYS   = [\"total\", \"proj\", \"nll\", \"smooth\",\n              \"diversity\", \"isotropy\", \"boundary\"]\n    COLORS = [\"black\", \"tab:orange\", \"tab:blue\", \"tab:green\",\n              \"tab:red\",   \"tab:purple\", \"tab:brown\"]\n    TITLES = [\"Total loss\", \"Projection (data fidelity)\",\n              \"Ensemble NLL\", \"TV smoothness\",\n              \"Ensemble diversity\", \"Isotropy prior\", \"Boundary\"]\n\n    present = [(k, c, t) for k, c, t in zip(KEYS, COLORS, TITLES) if k in hist and len(hist[k]) > 0]\n    n_plots = len(present)\n\n    if n_plots == 0:\n        fig, ax = plt.subplots()\n        ax.text(0.5, 0.5, \"No loss history available\",\n                ha=\"center\", va=\"center\", transform=ax.transAxes)\n        return fig\n\n    ncols = min(n_plots, 4)\n    nrows = (n_plots + ncols - 1) // ncols\n    fig, axes = plt.subplots(nrows, ncols,\n                              figsize=(5 * ncols, 3.8 * nrows),\n                              squeeze=False)\n\n    for idx, (key, color, title) in enumerate(present):\n        ax  = axes[idx // ncols][idx % ncols]\n        arr = np.array(hist[key], dtype=np.float64)\n        xs  = np.arange(len(arr))\n\n        # Optionally clip extreme values for readability\n        finite = arr[np.isfinite(arr)]\n        if len(finite) == 0:\n            ax.set_visible(False)\n            continue\n\n        ax.semilogy(xs, np.where(np.isfinite(arr), arr, np.nan),\n                    lw=0.7, alpha=0.85, color=color)\n\n        # Rolling mean (window = 5% of length)\n        win = max(1, len(arr) // 20)\n        if len(arr) >= win:\n            rm = np.convolve(finite, np.ones(win) / win, mode=\"valid\")\n            xs_rm = np.linspace(win // 2, len(arr) - win // 2, len(rm))\n            ax.semilogy(xs_rm, rm, lw=1.8, color=color, alpha=0.5, label=\"rolling mean\")\n\n        ax.set_title(title, fontsize=10)\n        ax.set_xlabel(\"step\")\n        ax.set_ylabel(\"loss (log scale)\")\n        ax.grid(True, alpha=0.3)\n        ax.legend(fontsize=8)\n\n    # Hide unused axes\n    for idx in range(n_plots, nrows * ncols):\n        axes[idx // ncols][idx % ncols].set_visible(False)\n\n    fig.suptitle(\"VICTOR v6.0 \u2014 Training loss curves\",\n                 fontsize=13, fontweight=\"bold\")\n    plt.tight_layout()\n\n    if save_path is not None:\n        fig.savefig(save_path, dpi=150, bbox_inches=\"tight\")\n        print(f\"Loss curves saved \u2192 {save_path}\")\n\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 8.  Summary metrics bar chart\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_metrics_summary(eval_bundles: list) -> plt.Figure:\n    \"\"\"\n    Bar charts of PSNR, CC, and RelErr across all evaluated profiles.\n\n    Parameters\n    ----------\n    eval_bundles : list[EvalBundle]\n\n    Returns\n    -------\n    matplotlib Figure\n    \"\"\"\n    idxs    = [b.metrics.profile_idx for b in eval_bundles]\n    psnrs   = [b.metrics.psnr        for b in eval_bundles]\n    ccs     = [b.metrics.cc          for b in eval_bundles]\n    rel_errs= [b.metrics.rel_err     for b in eval_bundles]\n\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n    x = np.arange(len(idxs))\n    w = 0.65\n\n    axes[0].bar(x, psnrs, width=w, color=\"steelblue\", alpha=0.85)\n    axes[0].set_title(\"PSNR [dB]  (\u2191 better)\", fontsize=11)\n    axes[0].set_xlabel(\"Profile\")\n    axes[0].set_xticks(x); axes[0].set_xticklabels(idxs)\n    axes[0].axhline(np.mean(psnrs), color=\"red\", lw=1.2,\n                    linestyle=\"--\", label=f\"mean={np.mean(psnrs):.1f} dB\")\n    axes[0].legend(fontsize=9); axes[0].grid(True, axis=\"y\", alpha=0.3)\n\n    axes[1].bar(x, ccs, width=w, color=\"darkorange\", alpha=0.85)\n    axes[1].set_title(\"Cross-Correlation  (\u2191 better)\", fontsize=11)\n    axes[1].set_xlabel(\"Profile\")\n    axes[1].set_xticks(x); axes[1].set_xticklabels(idxs)\n    axes[1].axhline(np.mean(ccs), color=\"red\", lw=1.2,\n                    linestyle=\"--\", label=f\"mean={np.mean(ccs):.3f}\")\n    axes[1].set_ylim(0, 1.05)\n    axes[1].legend(fontsize=9); axes[1].grid(True, axis=\"y\", alpha=0.3)\n\n    axes[2].bar(x, rel_errs, width=w, color=\"forestgreen\", alpha=0.85)\n    axes[2].set_title(\"Relative L2 Error  (\u2193 better)\", fontsize=11)\n    axes[2].set_xlabel(\"Profile\")\n    axes[2].set_xticks(x); axes[2].set_xticklabels(idxs)\n    axes[2].axhline(np.mean(rel_errs), color=\"red\", lw=1.2,\n                    linestyle=\"--\", label=f\"mean={np.mean(rel_errs):.3f}\")\n    axes[2].legend(fontsize=9); axes[2].grid(True, axis=\"y\", alpha=0.3)\n\n    fig.suptitle(\"VICTOR v6.0 \u2014 Evaluation Metrics Summary\",\n                 fontsize=13, fontweight=\"bold\")\n    plt.tight_layout()\n    return fig\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 9.  Per-profile forward evaluation\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef evaluate_profile(\n    model,\n    params    : dict,\n    profile   : dict,\n    w_ops,\n    grids,\n    rho_graph,\n) -> EvalBundle:\n    \"\"\"\n    Run a full forward pass on one profile and collect evaluation data.\n\n    Parameters\n    ----------\n    model     : VICTOR_v6 instance\n    params    : trained param tree\n    profile   : dict from data_loader.load_profiles()\n    w_ops     : WOperators\n    grids     : PixelGrids\n    rho_graph : RhoGraph\n\n    Returns\n    -------\n    EvalBundle\n    \"\"\"\n    import config as cfg\n\n    N = cfg.N_GRID\n    M = cfg.N_ENS\n\n    # Forward pass\n    eps_out, mean, std, preds = model.apply(\n        params,\n        grids.R_PIX,\n        grids.Z_PIX,\n        profile[\"psi_n\"],\n        profile[\"bpol_n\"],\n        rho_graph.EDGES_SRC,\n        rho_graph.EDGES_DST,\n        rho_graph.EDGE_W,\n        rho_graph.NODE_DEG,\n        grids.RHO_2D,\n    )\n\n    eps_pred_np = safe_numpy(eps_out)            # (N, N)\n    mean_np     = safe_numpy(mean).reshape(N, N) # (N, N)\n    std_np      = safe_numpy(std).reshape(N, N)  # (N, N)\n    preds_np    = safe_numpy(preds).reshape(M, N, N)  # (M, N, N)\n\n    eps_gt_np   = profile[\"eps_n\"]               # (N, N)  already numpy\n\n    # Projected sinogram from prediction\n    g_pred_jax  = w_ops.matvec(eps_out.flatten())\n    g_pred_np   = safe_numpy(g_pred_jax)\n    g_gt_np     = safe_numpy(profile[\"g_ideal\"])\n\n    # Active mask (all ones in eval mode \u2014 same treatment as verify_losses)\n    active_mask = np.ones(128, dtype=np.float32)\n\n    metrics = compute_all_metrics(\n        eps_pred    = eps_pred_np,\n        eps_gt      = eps_gt_np,\n        g_pred      = g_pred_np,\n        g_gt        = g_gt_np,\n        active_mask = active_mask,\n        profile_idx = int(profile.get(\"idx\", 0)),\n    )\n\n    return EvalBundle(\n        profile_idx = int(profile.get(\"idx\", 0)),\n        eps_pred    = eps_pred_np,\n        eps_gt      = eps_gt_np,\n        mean_2d     = mean_np,\n        std_2d      = std_np,\n        preds_2d    = preds_np,\n        g_pred      = g_pred_np,\n        g_gt        = g_gt_np,\n        metrics     = metrics,\n    )\n\n\ndef evaluate_all(\n    model,\n    params   : dict,\n    profiles : list,\n    w_ops,\n    grids,\n    rho_graph,\n) -> list:\n    \"\"\"\n    Evaluate all profiles and return a list of EvalBundles.\n\n    Parameters\n    ----------\n    model, params, w_ops, grids, rho_graph : as per evaluate_profile()\n    profiles : list of profile dicts\n\n    Returns\n    -------\n    list[EvalBundle]\n    \"\"\"\n    bundles = []\n    print(f\"\\nEvaluating {len(profiles)} profile(s)...\")\n    for i, prof in enumerate(profiles):\n        eb = evaluate_profile(model, params, prof, w_ops, grids, rho_graph)\n        m  = eb.metrics\n        print(\n            f\"  Profile {m.profile_idx:3d}: \"\n            f\"PSNR={m.psnr:6.2f} dB  \"\n            f\"CC={m.cc:.4f}  \"\n            f\"RelErr={m.rel_err:.4f}  \"\n            f\"ProjMSE={m.proj_mse:.2e}\"\n        )\n        bundles.append(eb)\n\n    # Print aggregate\n    psnrs   = [b.metrics.psnr    for b in bundles]\n    ccs     = [b.metrics.cc      for b in bundles]\n    rerrs   = [b.metrics.rel_err for b in bundles]\n    print(f\"\\n  \u2500\u2500 Aggregate (n={len(bundles)}) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\")\n    print(f\"     PSNR    : {np.mean(psnrs):.2f} \u00b1 {np.std(psnrs):.2f} dB\")\n    print(f\"     CC      : {np.mean(ccs):.4f} \u00b1 {np.std(ccs):.4f}\")\n    print(f\"     RelErr  : {np.mean(rerrs):.4f} \u00b1 {np.std(rerrs):.4f}\")\n    return bundles\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 10.  Convenience: evaluate all profiles + save all figures\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef plot_all_profiles(\n    model,\n    params      : dict,\n    profiles    : list,\n    w_ops,\n    grids,\n    rho_graph,\n    results_dir : str = cfg.RESULTS_DIR,\n    hist        : Optional[dict] = None,\n) -> list:\n    \"\"\"\n    Full evaluation pipeline for all profiles:\n      1. Runs evaluate_all()\n      2. Saves reconstruction, sinogram, radial, uncertainty, and ensemble\n         figures for every profile into results_dir/\n      3. Saves a metrics-summary bar chart\n      4. Optionally saves loss curves (if hist is provided)\n      5. Returns list[EvalBundle]\n\n    Parameters\n    ----------\n    model, params, profiles, w_ops, grids, rho_graph : pipeline objects\n    results_dir : str   where to write PNG files (created if absent)\n    hist        : dict, optional   training history from trainer.train()\n\n    Returns\n    -------\n    list[EvalBundle]\n    \"\"\"\n    os.makedirs(results_dir, exist_ok=True)\n\n    # \u2500\u2500 Run evaluation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    eval_bundles = evaluate_all(model, params, profiles, w_ops, grids, rho_graph)\n\n    # \u2500\u2500 Per-profile figures \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    active_mask = np.ones(128, dtype=np.float32)\n\n    for eb in eval_bundles:\n        pid = eb.profile_idx\n\n        # Reconstruction (4-panel)\n        fig = plot_reconstruction(\n            eps_pred    = eb.eps_pred,\n            eps_gt      = eb.eps_gt,\n            std_2d      = eb.std_2d,\n            rho_2d      = grids.RHO_2D,\n            R_pix       = grids.R_PIX,\n            Z_pix       = grids.Z_PIX,\n            metrics     = eb.metrics,\n            profile_idx = pid,\n        )\n        path = os.path.join(results_dir, f\"reconstruction_profile_{pid:03d}.png\")\n        fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n        plt.close(fig)\n        print(f\"  Saved: {path}\")\n\n        # Sinogram residual\n        fig = plot_sinogram_residual(\n            g_pred      = eb.g_pred,\n            g_gt        = eb.g_gt,\n            active_mask = active_mask,\n            profile_idx = pid,\n        )\n        path = os.path.join(results_dir, f\"sinogram_profile_{pid:03d}.png\")\n        fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n        plt.close(fig)\n        print(f\"  Saved: {path}\")\n\n        # Radial profile\n        fig = plot_radial_profile(\n            eps_pred    = eb.eps_pred,\n            eps_gt      = eb.eps_gt,\n            std_2d      = eb.std_2d,\n            rho_flat    = grids.RHO_FLAT,\n            profile_idx = pid,\n        )\n        path = os.path.join(results_dir, f\"radial_profile_{pid:03d}.png\")\n        fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n        plt.close(fig)\n        print(f\"  Saved: {path}\")\n\n        # Uncertainty map\n        fig = plot_uncertainty(\n            std_2d      = eb.std_2d,\n            rho_2d      = grids.RHO_2D,\n            R_pix       = grids.R_PIX,\n            Z_pix       = grids.Z_PIX,\n            profile_idx = pid,\n        )\n        path = os.path.join(results_dir, f\"uncertainty_profile_{pid:03d}.png\")\n        fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n        plt.close(fig)\n        print(f\"  Saved: {path}\")\n\n        # Ensemble members\n        fig = plot_ensemble_members(\n            preds_2d    = eb.preds_2d,\n            R_pix       = grids.R_PIX,\n            Z_pix       = grids.Z_PIX,\n            profile_idx = pid,\n        )\n        path = os.path.join(results_dir, f\"ensemble_members_profile_{pid:03d}.png\")\n        fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n        plt.close(fig)\n        print(f\"  Saved: {path}\")\n\n    # \u2500\u2500 Metrics summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    fig  = plot_metrics_summary(eval_bundles)\n    path = os.path.join(results_dir, \"metrics_summary.png\")\n    fig.savefig(path, dpi=150, bbox_inches=\"tight\")\n    plt.close(fig)\n    print(f\"  Saved: {path}\")\n\n    # \u2500\u2500 Loss curves (optional) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    if hist is not None:\n        path = os.path.join(results_dir, \"loss_curves.png\")\n        fig  = plot_loss_curves(hist, save_path=path)\n        plt.close(fig)\n\n    # \u2500\u2500 CSV summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    csv_path = os.path.join(results_dir, \"metrics_summary.csv\")\n    save_summary_csv(eval_bundles, csv_path)\n\n    print(f\"\\nAll figures saved to: {results_dir}\")\n    return eval_bundles\n\n\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n# 11.  CSV export\n# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\n\ndef save_summary_csv(eval_bundles: list, path: str) -> None:\n    \"\"\"\n    Write a CSV table of per-profile metrics.\n\n    Columns: profile_idx, psnr_dB, cc, rel_err, proj_mse\n\n    Parameters\n    ----------\n    eval_bundles : list[EvalBundle]\n    path         : str   destination file path\n    \"\"\"\n    import csv\n\n    rows = [(\"profile_idx\", \"psnr_dB\", \"cc\", \"rel_err\", \"proj_mse\")]\n    for eb in eval_bundles:\n        m = eb.metrics\n        rows.append((\n            m.profile_idx,\n            round(m.psnr,    4),\n            round(m.cc,      6),\n            round(m.rel_err, 6),\n            round(m.proj_mse,8),\n        ))\n\n    with open(path, \"w\", newline=\"\") as f:\n        writer = csv.writer(f)\n        writer.writerows(rows)\n\n    print(f\"Metrics CSV saved \u2192 {path}\")\n\n\n# \u2500\u2500 Module self-test \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nif __name__ == \"__main__\":\n    \"\"\"Quick sanity check with random arrays (no model or data needed).\"\"\"\n    N, M = cfg.N_GRID, cfg.N_ENS\n    rng  = np.random.default_rng(0)\n\n    pred = rng.random((N, N)).astype(np.float32)\n    gt   = rng.random((N, N)).astype(np.float32)\n    std  = rng.random((N, N)).astype(np.float32) * 0.05\n\n    lin      = np.linspace(-cfg.EXT, cfg.EXT, N)\n    XX, YY   = np.meshgrid(lin, lin)\n    rho_2d   = np.sqrt((XX / cfg.AP)**2 + (YY / cfg.BP)**2).astype(np.float32)\n    R_pix    = (cfg.R0 + XX).flatten().astype(np.float32)\n    Z_pix    = YY.flatten().astype(np.float32)\n    rho_flat = rho_2d.flatten()\n\n    psnr    = compute_psnr(pred, gt)\n    cc      = compute_cc(pred, gt)\n    rel_err = compute_rel_err(pred, gt)\n    print(f\"PSNR={psnr:.2f} dB  CC={cc:.4f}  RelErr={rel_err:.4f}\")\n\n    g_pred = rng.random(128).astype(np.float32)\n    g_gt   = rng.random(128).astype(np.float32)\n    mask   = np.ones(128, np.float32)\n\n    m = compute_all_metrics(pred, gt, g_pred, g_gt, mask, profile_idx=0)\n    print(m)\n\n    fig = plot_reconstruction(pred, gt, std, rho_2d, R_pix, Z_pix, m, 0)\n    plt.close(fig); print(\"plot_reconstruction OK\")\n\n    fig = plot_sinogram_residual(g_pred, g_gt, mask, 0)\n    plt.close(fig); print(\"plot_sinogram_residual OK\")\n\n    fig = plot_radial_profile(pred, gt, std, rho_flat, 0)\n    plt.close(fig); print(\"plot_radial_profile OK\")\n\n    fig = plot_uncertainty(std, rho_2d, R_pix, Z_pix, 0)\n    plt.close(fig); print(\"plot_uncertainty OK\")\n\n    preds_2d = rng.random((M, N, N)).astype(np.float32)\n    fig = plot_ensemble_members(preds_2d, R_pix, Z_pix, 0)\n    plt.close(fig); print(\"plot_ensemble_members OK\")\n\n    hist_dummy = {\n        \"total\": list(np.exp(-np.linspace(0, 5, 300))),\n        \"proj\":  list(np.exp(-np.linspace(0, 5, 300)) * 0.8),\n        \"nll\":   list(np.exp(-np.linspace(0, 4, 300)) * 0.3),\n    }\n    fig = plot_loss_curves(hist_dummy)\n    plt.close(fig); print(\"plot_loss_curves OK\")\n\n    print(\"\\nevaluate.py self-test PASSED\")\n"

print('Module sources loaded into memory.')

### Write modules to Drive
Run this cell once after Cell 1 to copy all embedded module source files to `MODULES_DIR`.

In [ ]:
# ── Write embedded modules to MODULES_DIR ──────────────────
# Run this once after Cell 1 so that load_modules() can find them.

import os

_MODULE_SOURCES = {
    "config.py":      _SRC_config,
    "geometry.py":    _SRC_geometry,
    "data_loader.py": _SRC_data_loader,
    "model.py":       _SRC_model,
    "losses.py":      _SRC_losses,
    "trainer.py":     _SRC_trainer,
    "checkpoint.py":  _SRC_checkpoint,
    "evaluate.py":    _SRC_evaluate,
}

for fname, src in _MODULE_SOURCES.items():
    dst = os.path.join(MODULES_DIR, fname)
    with open(dst, 'w') as f:
        f.write(src)
    print(f'  Written: {dst}')

print('\nAll modules written to', MODULES_DIR)
modules = load_modules(force_reload=True)

---
## Cell 2 — Load W Matrix · Geometry · Profiles
*Calls: `geometry.py`, `data_loader.py`*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 2: Load W matrix + geometry + profiles
# ============================================================
# Depends on Cell 1 (installs, Drive mount, MODULES_DIR on sys.path).
# Calls: geometry.py, data_loader.py, config.py
# ============================================================

# ── Reload modules (picks up any Drive edits) ───────────────
modules = load_modules(force_reload=True)

cfg         = modules["config"]
geom        = modules["geometry"]
data_loader = modules["data_loader"]

# ── Propagate Drive path into config ────────────────────────
cfg.update_paths(DRIVE)

# ── Load W matrix ────────────────────────────────────────────
w_bundle = data_loader.load_W_matrix()

# Expose operators at cell scope for downstream cells
W_BCOO       = w_bundle.W_BCOO
ACTIVE_MASK  = w_bundle.ACTIVE_MASK
N_ACTIVE     = w_bundle.N_ACTIVE
W_matvec     = w_bundle.w_ops.matvec
W_vecmat     = w_bundle.w_ops.vecmat

# ── Build geometry ───────────────────────────────────────────
grids, rays, rho_graph, _ = geom.build_all_geometry()

# Expose geometry arrays at cell scope
RHO_2D     = grids.RHO_2D
RHO_FLAT   = grids.RHO_FLAT
THETA_FLAT = grids.THETA_FLAT
R_PIX      = grids.R_PIX
Z_PIX      = grids.Z_PIX

RAY_R      = rays.RAY_R
RAY_Z      = rays.RAY_Z
RAY_DS     = rays.RAY_DS
RAY_V      = rays.RAY_V
MAX_STEPS  = rays.MAX_STEPS

EDGES_SRC  = rho_graph.EDGES_SRC
EDGES_DST  = rho_graph.EDGES_DST
EDGE_W     = rho_graph.EDGE_W
NODE_DEG   = rho_graph.NODE_DEG

# ── Load profiles ─────────────────────────────────────────────
profiles = data_loader.load_profiles(
    w_bundle = w_bundle,
    grids    = grids,
)

# ── Noise helper (re-export for downstream cells) ─────────────
inject_noise = data_loader.inject_noise

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*50)
print(f"Cell 2 complete")
print(f"  W        : {w_bundle.W_norm.shape}  NNZ={w_bundle.W_norm.nnz}  active={N_ACTIVE}/128")
print(f"  Grid     : {RHO_2D.shape}  R∈[{float(R_PIX.min()):.2f},{float(R_PIX.max()):.2f}]m")
print(f"  Rays     : {RAY_R.shape[0]}×{MAX_STEPS}")
print(f"  Edges    : {len(EDGES_SRC)}")
print(f"  Profiles : {len(profiles)}/{cfg.N_PROFILES}")
print("="*50)


---
## Cell 3 — Build Model
*Calls: `model.py`*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 3: Build Model
# ============================================================
# Depends on Cell 2 (W matrix, geometry, profiles all loaded).
# Imports: model.py  (HashGrid, SIRENLayer, SO2Harmonics,
#                     SharedTrunk, MemberAdapter, PIGNO, VICTOR_v6)
# ============================================================

# ── Reload modules (picks up any Drive edits) ───────────────
modules = load_modules(force_reload=True)

cfg_mod = modules["config"]
model_mod = modules["model"]

# ── Build model from first-profile shapes ───────────────────
# Use profile[0] field arrays for init (same shape for all profiles).
_p0 = profiles[0]["psi_n"]    # (N_GRID²,) float32
_b0 = profiles[0]["bpol_n"]   # (N_GRID²,) float32

bundle = model_mod.build_model(
    R_flat   = R_PIX,
    Z_flat   = Z_PIX,
    psi_n    = _p0,
    bpol_n   = _b0,
    esrc     = EDGES_SRC,
    edst     = EDGES_DST,
    ew       = EDGE_W,
    ndeg     = NODE_DEG,
    rho_2d   = RHO_2D,
    seed     = 0,
)

# ── Verify: shapes + NaN check ──────────────────────────────
model_mod.verify_model(
    bundle,
    R_PIX, Z_PIX, _p0, _b0,
    EDGES_SRC, EDGES_DST, EDGE_W, NODE_DEG, RHO_2D,
)

# ── Expose at cell scope for downstream cells ────────────────
model  = bundle.model
params = bundle.params

print("\n" + "="*50)
print("Cell 3 complete — model ready")
print("="*50)


---
## Cell 4 — Build Loss Functions
*Calls: `losses.py`*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 4: Build Loss Functions
# ============================================================
# Depends on Cell 3 (model, params, w_bundle, grids, rho_graph,
#                    profiles all in scope).
# Imports: losses.py  (loss_projection, loss_boundary,
#                      loss_smoothness, loss_ensemble_nll,
#                      loss_ensemble_diversity, loss_isotropy,
#                      loss_positivity, loss_fn, verify_losses)
# ============================================================

# ── Reload modules (picks up any Drive edits) ───────────────
modules = load_modules(force_reload=True)

losses_mod = modules["losses"]

# ── Expose individual loss functions at cell scope ───────────
loss_projection      = losses_mod.loss_projection
loss_boundary        = losses_mod.loss_boundary
loss_smoothness      = losses_mod.loss_smoothness
loss_ensemble_nll    = losses_mod.loss_ensemble_nll
loss_ensemble_diversity = losses_mod.loss_ensemble_diversity
loss_isotropy        = losses_mod.loss_isotropy
loss_positivity      = losses_mod.loss_positivity
loss_fn              = losses_mod.loss_fn
LossWeights          = losses_mod.LossWeights
DEFAULT_WEIGHTS      = losses_mod.DEFAULT_WEIGHTS

# ── Verify all loss components on profile[0] ─────────────────
losses_mod.verify_losses(
    model     = model,
    params    = params,
    profile   = profiles[0],
    w_ops     = w_bundle.w_ops,
    grids     = grids,
    rho_graph = rho_graph,
)

# ── Print active loss weights ─────────────────────────────────
print("\nActive loss weights:")
for field, val in DEFAULT_WEIGHTS._asdict().items():
    print(f"  {field:<18}: {val}")

print("\n" + "="*50)
print("Cell 4 complete — loss functions ready")
print("="*50)


---
## Cell 5 — Training Loop
*Calls: `trainer.py`, `checkpoint.py`*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 5: Training Loop
# ============================================================
# Depends on:
#   Cell 1 — installs, Drive mount, MODULES_DIR on sys.path,
#             CKPT_DIR, DRIVE, LOG_EVERY, SAVE_EVERY constants
#   Cell 2 — W matrix, geometry, profiles in cell scope
#   Cell 3 — model, params built
#   Cell 4 — loss_fn, LossWeights, DEFAULT_WEIGHTS imported
#
# Modules consumed:
#   trainer.py    — build_optimizer, make_train_step, train
#   checkpoint.py — build_ckpt_manager, do_checkpoint, resume
# ============================================================

# ── Reload all modules (picks up any Drive edits) ───────────────────
modules = load_modules(force_reload=True)

trainer_mod  = modules["trainer"]
ckpt_mod     = modules["checkpoint"]
cfg_mod      = modules["config"]
losses_mod   = modules["losses"]

# ── Unpack conveniences ─────────────────────────────────────────────
build_optimizer   = trainer_mod.build_optimizer
make_train_step   = trainer_mod.make_train_step
train             = trainer_mod.train

build_ckpt_manager = ckpt_mod.build_ckpt_manager
do_checkpoint      = ckpt_mod.do_checkpoint
resume             = ckpt_mod.resume

DEFAULT_WEIGHTS    = losses_mod.DEFAULT_WEIGHTS

# ── Optimiser ────────────────────────────────────────────────────────
# Total steps: N_EPOCHS steps per profile, one full pass through all profiles.
# The schedule is shared across profiles so the LR keeps decaying globally.
total_steps = cfg_mod.N_EPOCHS * len(profiles)

tx = build_optimizer(
    total_steps = total_steps,
    lr          = cfg_mod.LR,          # 3e-4
    warmup      = 500,
    lr_end      = 5e-5,
)

# ── JIT-compiled training step ───────────────────────────────────────
# make_train_step closes over model and w_ops so neither is passed as
# a JIT argument (JAX cannot trace nn.Module or WOperators objects).
step_fn = make_train_step(
    model   = model,
    w_ops   = w_bundle.w_ops,
    weights = DEFAULT_WEIGHTS,
)

# ── Initialise fresh state ──────────────────────────────────────────
params_init = params                      # from Cell 3 build_model()
opt_state   = tx.init(params_init)

# ── CheckpointManager ────────────────────────────────────────────────
ckpt_mgr = build_ckpt_manager(CKPT_DIR, max_to_keep=3)

# ── Checkpoint save callback (passed into the training loop) ─────────
def _do_checkpoint(ep_global, prof_idx, stage_idx, ep_in_stage, best_data):
    """Thin wrapper so the training loop stays decoupled from CKPT_DIR."""
    do_checkpoint(
        mgr         = ckpt_mgr,
        params      = params,      # captured from outer scope (updated in-place)
        opt_state   = opt_state,   # same
        ep_global   = ep_global,
        prof_idx    = prof_idx,
        stage_idx   = stage_idx,
        ep_in_stage = ep_in_stage,
        best_data   = best_data,
        ckpt_dir    = CKPT_DIR,
    )

# NOTE: params and opt_state are reassigned by train() — the callback
# must reference them through the module-level names below (see the
# "mutable closure" pattern: reassign at cell scope, callback reads them).

# ── Resume from checkpoint (or start fresh) ──────────────────────────
bundle = resume(
    mgr       = ckpt_mgr,
    ckpt_dir  = CKPT_DIR,
    params    = params_init,
    opt_state = opt_state,
    tx        = tx,
)

params    = bundle.params
opt_state = bundle.opt_state

# ── Report parameter count ───────────────────────────────────────────
import jax
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params:,}")

# ── Training ─────────────────────────────────────────────────────────
# The loop mutates params and opt_state and returns them at the end.
# The do_checkpoint callback is a closure that reads the CURRENT values
# of these variables — we update the cell-scope names after each call
# by reassigning from the returned tuple.

# History dict persists across re-runs in the same session so loss
# curves accumulate without gaps.
try:
    hist
except NameError:
    hist = {}

params, opt_state, hist, best_data = train(
    step_fn          = step_fn,
    params           = params,
    opt_state        = opt_state,
    tx               = tx,
    profiles         = profiles,
    R_flat           = R_PIX,
    Z_flat           = Z_PIX,
    esrc             = EDGES_SRC,
    edst             = EDGES_DST,
    ew               = EDGE_W,
    ndeg             = NODE_DEG,
    rho_2d           = RHO_2D,
    rho_flat         = RHO_FLAT,
    active_mask      = ACTIVE_MASK.astype(float),
    inject_noise     = inject_noise,
    stages           = cfg_mod.STAGES,
    ep_global        = bundle.ep_global,
    best_data        = bundle.best_data,
    start_prof       = bundle.start_prof,
    start_stage      = bundle.start_stage,
    start_ep         = bundle.start_ep,
    hist             = hist,
    do_checkpoint_fn = _do_checkpoint,
    log_every        = LOG_EVERY,
    save_every       = SAVE_EVERY,
)

# ── Final checkpoint ─────────────────────────────────────────────────
ep_global = bundle.ep_global + sum(s for s, _ in cfg_mod.STAGES) * len(profiles)
do_checkpoint(
    mgr         = ckpt_mgr,
    params      = params,
    opt_state   = opt_state,
    ep_global   = ep_global,
    prof_idx    = len(profiles),
    stage_idx   = 0,
    ep_in_stage = 0,
    best_data   = best_data,
    ckpt_dir    = CKPT_DIR,
)

# ── Quick loss-curve plot ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("VICTOR v6.0 — Training curves", fontweight="bold")

# Total loss
if "total" in hist and hist["total"]:
    axes[0].semilogy(hist["total"], lw=0.7, alpha=0.8)
    axes[0].set_title("Total loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss (log)")
    axes[0].grid(True, alpha=0.3)

# Projection loss
if "proj" in hist and hist["proj"]:
    axes[1].semilogy(hist["proj"], lw=0.7, alpha=0.8, color="tab:orange")
    axes[1].set_title("Projection (data-fidelity) loss")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("loss (log)")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150)
plt.show()
print(f"Curves saved → {RESULTS_DIR}/training_curves.png")

print("\n" + "="*50)
print("Cell 5 complete — training finished")
print(f"  Best projection loss : {best_data:.6f}")
print(f"  Checkpoint dir       : {CKPT_DIR}")
print("="*50)


---
## Cell 6 — Evaluation + Plots
*Calls: `evaluate.py`*

In [ ]:
# ============================================================
# VICTOR v6.0 — Cell 6: Evaluation + Plots
# ============================================================
# Depends on:
#   Cell 1 — installs, Drive mount, MODULES_DIR, RESULTS_DIR,
#             load_modules() in scope
#   Cell 2 — W matrix, geometry, profiles in cell scope
#   Cell 3 — model, params in cell scope
#   Cell 5 — hist, best_data from training (optional)
#
# Modules consumed:
#   evaluate.py — all metrics and plot functions
#
# What this cell does
# -------------------
#  1. Reloads all modules (picks up any Drive edits)
#  2. Runs a full forward pass on every loaded profile
#  3. Computes PSNR, CC, RelErr, sinogram MSE for each profile
#  4. Saves per-profile figures to RESULTS_DIR:
#       reconstruction_profile_NNN.png  — 4-panel (GT|Pred|Err|Uncert)
#       sinogram_profile_NNN.png        — measured vs predicted + residual
#       radial_profile_NNN.png          — ε vs ρ with uncertainty band
#       uncertainty_profile_NNN.png     — ensemble σ map
#       ensemble_members_profile_NNN.png— all M member predictions
#  5. Saves aggregate figures:
#       metrics_summary.png             — PSNR / CC / RelErr bar charts
#       loss_curves.png                 — training history (if hist exists)
#       metrics_summary.csv             — machine-readable metrics table
#  6. Displays inline previews of selected plots
# ============================================================

# ── Reload modules ──────────────────────────────────────────
modules = load_modules(force_reload=True)

eval_mod = modules.get("evaluate")
if eval_mod is None:
    raise ImportError(
        "evaluate.py not found in MODULES_DIR.\n"
        f"Copy evaluate.py to {MODULES_DIR}/ and re-run this cell."
    )

# ── Expose evaluation functions at cell scope ─────────────
evaluate_profile   = eval_mod.evaluate_profile
evaluate_all       = eval_mod.evaluate_all
plot_all_profiles  = eval_mod.plot_all_profiles
plot_reconstruction= eval_mod.plot_reconstruction
plot_sinogram_residual = eval_mod.plot_sinogram_residual
plot_radial_profile    = eval_mod.plot_radial_profile
plot_uncertainty       = eval_mod.plot_uncertainty
plot_ensemble_members  = eval_mod.plot_ensemble_members
plot_loss_curves       = eval_mod.plot_loss_curves
plot_metrics_summary   = eval_mod.plot_metrics_summary
save_summary_csv       = eval_mod.save_summary_csv
EvalBundle             = eval_mod.EvalBundle
MetricBundle           = eval_mod.MetricBundle

# ── Geometry convenience objects ─────────────────────────────
# Constructed in Cell 2; referenced here by name.
# We re-use the cell-scope variables directly — no re-build needed.
import importlib
cfg_mod  = modules["config"]
geom_mod = modules["geometry"]

from types import SimpleNamespace

_grids = SimpleNamespace(
    RHO_2D    = RHO_2D,
    RHO_FLAT  = RHO_FLAT,
    THETA_FLAT= THETA_FLAT,
    R_PIX     = R_PIX,
    Z_PIX     = Z_PIX,
)

_rho_graph = SimpleNamespace(
    EDGES_SRC = EDGES_SRC,
    EDGES_DST = EDGES_DST,
    EDGE_W    = EDGE_W,
    NODE_DEG  = NODE_DEG,
)

# ── Training history (graceful fallback) ─────────────────────
try:
    _hist = hist          # from Cell 5 training
except NameError:
    _hist = None
    print("Note: 'hist' not found in cell scope — loss curves will be skipped.")

# ── Run full evaluation + save all figures ────────────────────
print(f"\nEvaluation output directory: {RESULTS_DIR}")
print("=" * 55)

eval_bundles = plot_all_profiles(
    model       = model,
    params      = params,
    profiles    = profiles,
    w_ops       = w_bundle.w_ops,
    grids       = _grids,
    rho_graph   = _rho_graph,
    results_dir = RESULTS_DIR,
    hist        = _hist,
)

# ── Inline preview: reconstruction for first 3 profiles ──────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

N_PREVIEW = min(3, len(eval_bundles))
print(f"\nInline previews ({N_PREVIEW} profiles):")

for eb in eval_bundles[:N_PREVIEW]:
    pid = eb.profile_idx

    # Reconstruction 4-panel
    fig = plot_reconstruction(
        eps_pred    = eb.eps_pred,
        eps_gt      = eb.eps_gt,
        std_2d      = eb.std_2d,
        rho_2d      = RHO_2D,
        R_pix       = R_PIX,
        Z_pix       = Z_PIX,
        metrics     = eb.metrics,
        profile_idx = pid,
    )
    plt.show()

    # Radial profile
    fig = plot_radial_profile(
        eps_pred    = eb.eps_pred,
        eps_gt      = eb.eps_gt,
        std_2d      = eb.std_2d,
        rho_flat    = RHO_FLAT,
        profile_idx = pid,
    )
    plt.show()

    # Sinogram residual
    fig = plot_sinogram_residual(
        g_pred      = eb.g_pred,
        g_gt        = eb.g_gt,
        active_mask = ACTIVE_MASK.astype(float),
        profile_idx = pid,
    )
    plt.show()

# ── Inline preview: metrics summary ─────────────────────────
fig = plot_metrics_summary(eval_bundles)
plt.show()

# ── Inline preview: loss curves (if available) ───────────────
if _hist is not None and any(len(v) > 0 for v in _hist.values()):
    fig = plot_loss_curves(_hist)
    plt.show()
else:
    print("Loss curve plot skipped (no training history in scope).")

# ── Final summary table ──────────────────────────────────────
import numpy as np

print("\n" + "=" * 60)
print("VICTOR v6.0 — Evaluation Summary")
print("=" * 60)
print(f"  {'Profile':>8}  {'PSNR [dB]':>10}  {'CC':>8}  {'RelErr':>8}  {'ProjMSE':>10}")
print(f"  {'-'*8}  {'-'*10}  {'-'*8}  {'-'*8}  {'-'*10}")

for eb in eval_bundles:
    m = eb.metrics
    print(
        f"  {m.profile_idx:>8d}  "
        f"{m.psnr:>10.2f}  "
        f"{m.cc:>8.4f}  "
        f"{m.rel_err:>8.4f}  "
        f"{m.proj_mse:>10.2e}"
    )

psnrs   = [b.metrics.psnr    for b in eval_bundles]
ccs     = [b.metrics.cc      for b in eval_bundles]
rerrs   = [b.metrics.rel_err for b in eval_bundles]

print(f"  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*8}  {'─'*10}")
print(
    f"  {'mean':>8}  "
    f"{np.mean(psnrs):>10.2f}  "
    f"{np.mean(ccs):>8.4f}  "
    f"{np.mean(rerrs):>8.4f}"
)
print(
    f"  {'std':>8}  "
    f"{np.std(psnrs):>10.2f}  "
    f"{np.std(ccs):>8.4f}  "
    f"{np.std(rerrs):>8.4f}"
)
print("=" * 60)
print(f"\nAll figures and CSV saved to: {RESULTS_DIR}")
print("=" * 60)
print("Cell 6 complete — evaluation finished")
print("=" * 60)
